> The monitoring data and building-specific configuration are confidential and are not included. See `README.md` for execution requirements. Clear any generated outputs before sharing an executed notebook.


In [ ]:
# Private workspace and configuration
from copy import deepcopy as _privacy_deepcopy
from functools import lru_cache as _privacy_lru_cache
import json
import os
from pathlib import Path


def _private_config_path(variable):
    value = os.environ.get(variable)
    if not value:
        raise RuntimeError(f"Set {variable} before running this notebook; see README.md.")
    return Path(value).expanduser().resolve()


def private_workspace():
    """Return the configured private input/output directory."""
    path = _private_config_path("HVAC_PRIVATE_WORKSPACE")
    if not path.is_dir():
        raise FileNotFoundError("The configured private workspace does not exist.")
    return path


def _decode_private_values(value):
    if isinstance(value, list):
        return [_decode_private_values(item) for item in value]
    if isinstance(value, dict):
        if set(value) == {"__tuple_key_dict__"}:
            return {tuple(key): _decode_private_values(item)
                    for key, item in value["__tuple_key_dict__"]}
        return {key: _decode_private_values(item) for key, item in value.items()}
    return value


@_privacy_lru_cache(maxsize=1)
def _load_private_values():
    with _private_config_path("HVAC_PRIVATE_SETTINGS").open(encoding="utf-8") as handle:
        values = json.load(handle).get("private_values")
    if not isinstance(values, dict):
        raise ValueError("Private settings must contain a private_values object.")
    return _decode_private_values(values)


def private_value(key):
    """Read a private setting without embedding its value in the public source."""
    values = _load_private_values()
    if key not in values or values[key] is None:
        raise KeyError(f"Required private setting is missing: {key!r}.")
    return _privacy_deepcopy(values[key])


# Notebook 05: Neural component comparisons

This notebook compares the trained Shared NN from Notebook 02 with four controlled variants. All use the chronological split and evaluation procedure established in Notebooks 01–03. The original checkpoint remains the reference model.

| Variant | Change from the reference model |
|---|---|
| `No_q_weighting` | Use uniform weights in the training duration loss. |
| `No_duration_loss` | Remove the duration-loss training gradient while retaining duration-based validation selection. |
| `No_derivative_history` | Remove historical derivative lags while retaining current derivative estimates. |
| `Separate_phase_NNs` | Train separate heating and cooling networks with independent validation selection. |

The model inputs, training-only fitted transformations, target scales and evaluation cases are checked against the reference artifacts. Duration accuracy is the primary outcome; temperature and rate accuracy are secondary outcomes. In the code, `phase` denotes operating mode and an anchor denotes a duration prediction time.

These comparisons assess the specified interventions under this training procedure. Separate-mode networks also change total capacity and checkpoint selection, so that comparison does not isolate weight sharing alone. Offline input reconstruction remains as specified in Notebook 01.


In [ ]:


# Imports, saved reference paths, and ablation configuration

import os, re, json, math, random, warnings, shutil, hashlib, time
from pathlib import Path
from itertools import cycle
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
import joblib

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset
except Exception as e:
    raise RuntimeError('PyTorch is required for Notebook 05.') from e

try:
    from google.colab import drive  # type: ignore
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except Exception:
    pass

def safe_torch_load(path, map_location=None):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

warnings.filterwarnings('default')

# Reference configuration inherited from Notebook 02

RANDOM_SEED = 42
STRICT_DETERMINISM = False

def reset_seeds(seed=RANDOM_SEED):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

reset_seeds()

if STRICT_DETERMINISM:
    torch.use_deterministic_algorithms(True, warn_only=True)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
else:
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.set_float32_matmul_precision('high')
    if hasattr(torch.backends, 'cuda'):
        torch.backends.cuda.matmul.allow_tf32 = True
    props = torch.cuda.get_device_properties(0)
    GPU_MEMORY_GB = props.total_memory / 1024**3
    print('GPU:', props.name, f'({GPU_MEMORY_GB:.1f} GB)')
else:
    GPU_MEMORY_GB = 0.0

USE_AMP = bool(DEVICE.type == 'cuda')
AMP_DTYPE = torch.float16

BASE_DIR = Path(str(private_workspace()))
if 'PAPER-TEMP-NEW' in str(BASE_DIR):
    raise RuntimeError('STOP: Notebook 05 must use PAPER-TEMP, not PAPER-TEMP-NEW.')

DATASET_FOLDER = 'full_rampup_5min_feature_dataset_UNIFIED_DERIVATIVE_DURATION_AWARE'
DATA_DIR = BASE_DIR / DATASET_FOLDER
SPLIT_MAP_PATH = DATA_DIR / 'shared_event_split_map.csv'
SPLIT_SIGNATURE_PATH = DATA_DIR / 'shared_event_split_signature.json'

CANONICAL_TRAIN_RUN = (
    BASE_DIR /
    'training_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL' /
    'paper_final_q_weighted'
)
CANONICAL_MODEL_DIR = CANONICAL_TRAIN_RUN / 'models'
CANONICAL_BEST_PATH = CANONICAL_MODEL_DIR / 'unified_directed_rate_nn.pt'
CANONICAL_MANIFEST_PATH = (
    CANONICAL_TRAIN_RUN /
    'training_manifest_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL.json'
)

# Exact Notebook-03 run used by Notebook 04 and the current manuscript.
NOTEBOOK03_RUN_ID = private_value('shared.notebook03_run_id')
NOTEBOOK03_RUN = (
    BASE_DIR /
    'duration_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL' /
    NOTEBOOK03_RUN_ID
)
NB03_ALL_PRED_PATH = (
    NOTEBOOK03_RUN /
    'predictions' /
    'ALL_remaining_duration_predictions_with_relative_errors.csv'
)
NB03_MAIN_PATH = (
    NOTEBOOK03_RUN /
    'tables' /
    'PAPER_MAIN_duration_metrics_EXACT_COMMON_SUPPORT.csv'
)
NB03_MANIFEST_PATH = (
    NOTEBOOK03_RUN /
    'duration_manifest_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL.json'
)

# Persistent output permits Colab resume.
ABLATION_ROOT = BASE_DIR / 'ablation_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL'
RUN_NAME = 'paper_final_core_ablations'
OUT_ROOT = ABLATION_ROOT / RUN_NAME
COMMON_DIR = OUT_ROOT / 'common'
TABLE_DIR = OUT_ROOT / 'tables'
FIG_DIR = OUT_ROOT / 'figures'
PRED_DIR = OUT_ROOT / 'predictions'
AUDIT_DIR = OUT_ROOT / 'audit'
VARIANT_ROOT = OUT_ROOT / 'variants'
for p in [COMMON_DIR, TABLE_DIR, FIG_DIR, PRED_DIR, AUDIT_DIR, VARIANT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# Fixed reference scientific constants

PHASES = ['heating', 'cooling']
STEP_MIN = 5.0
END_BAND_C = 0.4
DURATION_MAX_HORIZON_BY_PHASE = {'heating': 185.0, 'cooling': 65.0}

NN_HIDDEN_DIMS = [128, 64, 32]
NN_DROPOUT = 0.10
NN_LR = 1e-3
NN_WEIGHT_DECAY = 1e-4
NN_BATCH_TRAJECTORIES_PER_PHASE = 256 if GPU_MEMORY_GB >= 10 else 128
NN_BATCH_DURATION_ANCHORS_PER_PHASE = 256 if GPU_MEMORY_GB >= 10 else 128
NN_MAX_EPOCHS = 500
NN_VALIDATION_EVERY = 2
NN_EARLY_STOP_PATIENCE_VALIDATIONS = 10
NN_SCHEDULER_PATIENCE_VALIDATIONS = 2
NN_SCHEDULER_FACTOR = 0.5
NN_MIN_LR = 1e-6
NN_MIN_DELTA = 1e-5

# Notebook-03 evaluation completion-time procedure constants.
EXTRAPOLATE_FINAL_N_POINTS = 4
EXTRAPOLATE_MIN_ABS_SLOPE_C_PER_MIN = 1e-4
EXTRAPOLATE_MAX_EXTRA_MIN_BY_PHASE = {'heating': 60.0, 'cooling': 30.0}

# Ablation execution controls.
VARIANTS_TO_RUN = [
    'No_q_weighting',
    'No_duration_loss',
    'No_derivative_history',
    'Separate_phase_NNs',
]
RESUME_TRAINING = True
RESET_VARIANTS = []  # e.g. ['No_q_weighting'] to deliberately restart only that variant
FORCE_CONTINUE_AFTER_EARLY_STOP = False

# Reference replay verifies that Notebook-05 hard inference reproduces Notebook 03.
STRICT_CANONICAL_REPLAY = True
CANONICAL_REPLAY_TOL_MIN = 1e-3

# Paired event-cluster bootstrap for ablation deltas.
N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 42
CI_LEVEL = 0.95

print('DEVICE:', DEVICE)
print('DATA_DIR:', DATA_DIR)
print('Canonical PAPER-FINAL run:', CANONICAL_TRAIN_RUN)
print('Notebook 03 run:', NOTEBOOK03_RUN)
print('Notebook 05 output:', OUT_ROOT)
print('Variants:', VARIANTS_TO_RUN)


## 1. Load the shared datasets and split

Read the long-format temperature/rate targets and dedicated duration anchors. Each event retains its saved chronological partition. Duration supervision uses the anchor tables rather than requiring a valid future temperature target.


In [ ]:


# Load reference data and verify the saved split

def norm_event_id(x):
    if pd.isna(x):
        raise RuntimeError('Missing event_id')
    s = str(x).strip()
    if re.fullmatch(r'[-+]?\d+\.0+', s):
        s = str(int(float(s)))
    return s

def load_phase_long(phase):
    phase_dir = DATA_DIR / phase
    main = phase_dir / f'{phase}_full_rampup_5min_samples_LONG.csv'
    by_room = sorted(
        (phase_dir / 'by_room').glob(f'*_{phase}_full_rampup_5min_LONG.csv')
    )
    frames = []
    if main.exists():
        frames.append(pd.read_csv(main))
    frames += [pd.read_csv(p) for p in by_room]
    if not frames:
        raise FileNotFoundError(f'No LONG data found for {phase}')

    d = pd.concat(frames, ignore_index=True, sort=False)
    d['phase'] = phase

    for c in [
        'timestamp', 'target_timestamp', 'rampup_start_time',
        'rampup_end_time', 'rampup_end_grid_time', 'extraction_end_time'
    ]:
        if c in d.columns:
            d[c] = pd.to_datetime(d[c], errors='coerce')

    d['event_id'] = d['event_id'].map(norm_event_id)
    d['room'] = d['room'].astype(str).str.strip()
    d['shared_event_uid'] = (
        d['phase'] + '|' + d['room'] + '|' + d['event_id']
    )

    key = [
        'phase', 'room', 'event_id', 'timestamp',
        'target_timestamp', 'horizon_min'
    ]
    return (
        d.drop_duplicates(
            subset=[c for c in key if c in d.columns],
            keep='first'
        )
        .reset_index(drop=True)
    )

split_map = pd.read_csv(SPLIT_MAP_PATH)
split_map['event_id'] = split_map['event_id'].map(norm_event_id)
split_map['room'] = split_map['room'].astype(str).str.strip()
split_map['shared_event_uid'] = (
    split_map['phase'].astype(str) + '|' +
    split_map['room'] + '|' +
    split_map['event_id']
)
split_map = split_map[
    ['shared_event_uid', 'split', 'split_block']
].drop_duplicates()

phase_frames = []
for phase in PHASES:
    d = load_phase_long(phase)
    d = d.merge(
        split_map,
        on='shared_event_uid',
        how='left',
        validate='many_to_one'
    )
    if d['split'].isna().any():
        raise RuntimeError(f'{phase}: LONG rows do not match authoritative split map.')

    d['event_key'] = d['shared_event_uid']
    d['target_context'] = (
        d['target_context']
        if 'target_context' in d.columns
        else pd.Series('inside_event', index=d.index)
    ).astype(str).str.strip().str.lower()

    phase_frames.append(d)

DATA = pd.concat(phase_frames, ignore_index=True, sort=False)
DATA['_row_id'] = np.arange(len(DATA), dtype=int)
DATA['use_for_temperature_training'] = DATA['target_context'].isin(
    ['inside_event', 'post_end_trend_continuation', 'post_end_buffer']
)

required_long = [
    'phase', 'room', 'event_key', 'split', 'timestamp',
    'target_timestamp', 'horizon_min', 'Room_T', 'target_Room_T',
    'thermal_mode_sign', 'Phase_Target_T_C',
    'Directed_Distance_to_Target_C',
    'Room_dTdt_now_bwd2_C_per_min',
    'target_directed_rate_C_per_min'
]
missing = [c for c in required_long if c not in DATA.columns]
if missing:
    raise RuntimeError(f'Notebook 01 LONG columns missing: {missing}')

for c in [
    'horizon_min', 'Room_T', 'target_Room_T', 'thermal_mode_sign',
    'Phase_Target_T_C', 'Directed_Distance_to_Target_C',
    'Room_dTdt_now_bwd2_C_per_min',
    'target_directed_rate_C_per_min'
]:
    DATA[c] = pd.to_numeric(DATA[c], errors='coerce')

# Saved duration anchors.
anchor_frames = []
for phase in PHASES:
    p = DATA_DIR / phase / f'{phase}_remaining_duration_anchors_5min.csv'
    if not p.exists():
        raise FileNotFoundError(p)

    a = pd.read_csv(p)
    a['phase'] = phase

    for c in [
        'timestamp', 'anchor_timestamp', 'feature_source_timestamp',
        'rampup_start_time', 'rampup_end_time', 'rampup_end_grid_time'
    ]:
        if c in a.columns:
            a[c] = pd.to_datetime(a[c], errors='coerce')

    a['event_id'] = a['event_id'].map(norm_event_id)
    a['room'] = a['room'].astype(str).str.strip()
    a['shared_event_uid'] = (
        a['phase'] + '|' + a['room'] + '|' + a['event_id']
    )
    a = a.merge(
        split_map,
        on='shared_event_uid',
        how='left',
        validate='many_to_one'
    )
    if a['split'].isna().any():
        raise RuntimeError(
            f'{phase}: duration anchors do not match authoritative split map.'
        )

    a['event_key'] = a['shared_event_uid']
    a['true_remaining_duration_min'] = pd.to_numeric(
        a.get('true_remaining_duration_min', a.get('remaining_duration_min')),
        errors='coerce'
    )
    a['remaining_trajectory_quality_score'] = pd.to_numeric(
        a['remaining_trajectory_quality_score'],
        errors='coerce'
    ).clip(0.0, 1.0)
    a['thermal_mode_sign'] = pd.to_numeric(
        a['thermal_mode_sign'], errors='coerce'
    )
    a['Phase_Target_T_C'] = pd.to_numeric(
        a['Phase_Target_T_C'], errors='coerce'
    )
    a['Room_T'] = pd.to_numeric(a['Room_T'], errors='coerce')
    a['asof_gap_min'] = (
        a['anchor_timestamp'] - a['feature_source_timestamp']
    ).dt.total_seconds() / 60.0
    a['asof_gap_min'] = a['asof_gap_min'].clip(lower=0.0)

    kind = (
        a['anchor_kind']
        if 'anchor_kind' in a.columns
        else pd.Series('anchor', index=a.index)
    ).astype(str)
    a['anchor_kind'] = kind
    a['anchor_key'] = (
        a['event_key'] + '|' +
        a['anchor_timestamp'].astype(str) + '|' +
        kind
    )
    a['Duration_Crossing_Threshold_C'] = (
        a['Phase_Target_T_C'] -
        a['thermal_mode_sign'] * END_BAND_C
    )
    anchor_frames.append(a)

ANCHORS = pd.concat(anchor_frames, ignore_index=True, sort=False)
ANCHORS = (
    ANCHORS
    .drop_duplicates('anchor_key', keep='first')
    .reset_index(drop=True)
)

required_anchor = [
    'phase', 'room', 'event_key', 'split', 'anchor_key',
    'anchor_timestamp', 'feature_source_timestamp', 'Room_T',
    'Phase_Target_T_C', 'thermal_mode_sign',
    'true_remaining_duration_min',
    'remaining_trajectory_quality_score', 'asof_gap_min'
]
missing = [c for c in required_anchor if c not in ANCHORS.columns]
if missing:
    raise RuntimeError(f'Notebook 01 duration-anchor columns missing: {missing}')

def canonical_split_signature(split_frame):
    z = split_frame.copy()
    if 'shared_event_uid' not in z.columns:
        z['shared_event_uid'] = (
            z['phase'].astype(str) + '|' +
            z['room'].astype(str).str.strip() + '|' +
            z['event_id'].astype(str).str.strip()
        )

    parts = z['shared_event_uid'].astype(str).str.split('|', n=2, expand=True)
    sig = pd.DataFrame({
        'phase': parts[0],
        'room': parts[1],
        'event_id': parts[2],
        'split': z['split'].astype(str),
        'split_block': z['split_block'].astype(str),
    }).drop_duplicates().sort_values(
        ['phase', 'room', 'event_id']
    ).reset_index(drop=True)

    canonical = sig.to_csv(index=False, lineterminator='\n')
    return hashlib.sha256(canonical.encode('utf-8')).hexdigest(), sig

SHARED_SPLIT_SHA256, SHARED_SPLIT_CANONICAL = canonical_split_signature(
    split_map
)

if SPLIT_SIGNATURE_PATH.exists():
    with open(SPLIT_SIGNATURE_PATH, 'r', encoding='utf-8') as f:
        expected_sig = json.load(f)
    if SHARED_SPLIT_SHA256 != expected_sig.get('sha256'):
        raise RuntimeError(
            'Shared split SHA256 differs from Notebook 01. STOP.'
        )

for label, frame in [('LONG', DATA), ('DURATION_ANCHORS', ANCHORS)]:
    if frame['split'].isna().any():
        raise RuntimeError(f'{label}: missing split assignments.')
    if not set(frame['split'].unique()).issubset(
        {'train', 'valid', 'test'}
    ):
        raise RuntimeError(f'{label}: unexpected split labels.')

print('LONG rows:', len(DATA))
print('Duration anchors:', len(ANCHORS))
print('Shared split SHA256:', SHARED_SPLIT_SHA256)
display(
    ANCHORS.groupby(['phase', 'split']).agg(
        anchors=('anchor_key', 'nunique'),
        events=('event_key', 'nunique'),
        rooms=('room', 'nunique')
    )
)


## 2. Verify the feature sets

Reconstruct the reference input order and compare it with the saved preprocessor. The reduced-history variant removes past room and water-supply derivative lags, while retaining current first- and second-derivative estimates. Raw temperature-history lags remain excluded from the model inputs.


In [ ]:


# Reference features and the reduced-history variant

DERIVATIVE_HISTORY_LAGS_MINUTES = list(range(5, 65, 5))
SOURCE_DERIVATIVE_HISTORY_LAGS_MINUTES = list(range(5, 30, 5))

COMMON_INPUTS = [
    'horizon_min',
    'Room_T', 'Room_V', 'Phase_Target_T_C',
    'Directed_Distance_to_Target_C',
    'T_ext', 'T_ave_other_rooms',
    'Thermal_Source_T_C', 'Directed_Source_Room_DeltaT_C',
    'Thermal_Input_Proxy',
    'thermal_mode_sign',
    'Room_dTdt_now_bwd2_C_per_min',
    'Room_d2Tdt2_now_bwd2_C_per_min2',
    'hour_decimal', 'day_of_year', 'day_of_week', 'month',
    'is_weekend',
    'doy_sin', 'doy_cos', 'dow_sin', 'dow_cos',
    'month_sin', 'month_cos',
    'solar_declination_proxy_deg', 'daylight_hours_proxy',
    'sunrise_hour_proxy', 'sunset_hour_proxy',
    'daylight_fraction_proxy',
    'time_since_sunrise_h_proxy',
    'time_to_sunset_h_proxy',
    'solar_elevation_proxy',
]
COMMON_INPUTS += [
    f'Room_dTdt_central_-{lag}min_C_per_min'
    for lag in DERIVATIVE_HISTORY_LAGS_MINUTES
]
COMMON_INPUTS += [
    f'Room_d2Tdt2_central_-{lag}min_C_per_min2'
    for lag in DERIVATIVE_HISTORY_LAGS_MINUTES
]
COMMON_INPUTS += [
    f'ThermalSource_dTdt_central_-{lag}min_C_per_min'
    for lag in SOURCE_DERIVATIVE_HISTORY_LAGS_MINUTES
]
COMMON_INPUTS += [
    f'ThermalSource_d2Tdt2_central_-{lag}min_C_per_min2'
    for lag in SOURCE_DERIVATIVE_HISTORY_LAGS_MINUTES
]

FORBIDDEN_EXACT = {
    'room', 'room_label', 'event_id', 'event_key', 'shared_event_uid',
    'target_Room_T', 'target_timestamp',
    'target_directed_rate_C_per_min',
    'target_Room_dTdt_central_C_per_min',
    'remaining_time_to_end_min', 'remaining_duration_min',
    'true_remaining_duration_min',
    'remaining_trajectory_quality_score',
    'remaining_forward_progress_C',
    'remaining_reverse_motion_C',
    'remaining_total_variation_C',
    'remaining_quality_n_steps',
    'rampup_start_time', 'rampup_end_time',
    'rampup_end_grid_time', 'extraction_end_time',
    'split', 'split_block',
}
forbidden_pattern = re.compile(
    r'^Room_T -\d+min$|^Room_T \+\d+min$|^room__',
    re.I
)

train_all = DATA[
    (DATA['split'] == 'train') &
    DATA['use_for_temperature_training']
].copy()
anchor_train_all = ANCHORS[ANCHORS['split'].eq('train')].copy()

selected = []
audit = []

for c in COMMON_INPUTS:
    if c in FORBIDDEN_EXACT or forbidden_pattern.search(c):
        audit.append({'feature': c, 'status': 'forbidden'})
        continue
    if c not in DATA.columns:
        audit.append({'feature': c, 'status': 'missing_LONG'})
        continue
    if c != 'horizon_min' and c not in ANCHORS.columns:
        audit.append({'feature': c, 'status': 'missing_duration_anchor'})
        continue

    tr = (
        pd.to_numeric(train_all[c], errors='coerce')
        .replace([np.inf, -np.inf], np.nan)
    )
    if tr.notna().sum() == 0:
        audit.append({'feature': c, 'status': 'all_missing_LONG_train'})
        continue

    if c != 'horizon_min':
        atr = (
            pd.to_numeric(anchor_train_all[c], errors='coerce')
            .replace([np.inf, -np.inf], np.nan)
        )
        if atr.notna().sum() == 0:
            audit.append({
                'feature': c,
                'status': 'all_missing_duration_anchor_train'
            })
            continue

    phase_counts = {
        p: int(
            pd.to_numeric(
                train_all.loc[train_all['phase'].eq(p), c],
                errors='coerce'
            )
            .replace([np.inf, -np.inf], np.nan)
            .notna().sum()
        )
        for p in PHASES
    }
    if any(v == 0 for v in phase_counts.values()):
        audit.append({
            'feature': c,
            'status': 'missing_in_one_phase_train',
            **{f'finite_{p}': v for p, v in phase_counts.items()}
        })
        continue

    if tr.dropna().nunique() <= 1 and c != 'thermal_mode_sign':
        audit.append({'feature': c, 'status': 'zero_variance_train'})
        continue

    selected.append(c)
    audit.append({
        'feature': c,
        'status': 'selected',
        'missing_fraction_train': float(tr.isna().mean()),
        **{f'finite_{p}': v for p, v in phase_counts.items()}
    })

if 'thermal_mode_sign' not in selected:
    selected.append('thermal_mode_sign')

if any(forbidden_pattern.search(c) for c in selected):
    raise RuntimeError('Raw room-temperature lag or room identity slipped in.')
if any(c in FORBIDDEN_EXACT for c in selected):
    raise RuntimeError('Forbidden target/identity input slipped in.')

FULL_MODEL_INPUTS = selected

# Reference preprocessor from the saved reference run.
CANONICAL_PREPROCESSOR = joblib.load(
    CANONICAL_MODEL_DIR / 'unified_numeric_preprocessor.joblib'
)
CANONICAL_MODEL_INPUTS = list(
    CANONICAL_PREPROCESSOR['numeric_cols']
)

if FULL_MODEL_INPUTS != CANONICAL_MODEL_INPUTS:
    raise RuntimeError(
        'Reconstructed PAPER-FINAL input order differs from the frozen '
        'canonical preprocessor.'
    )

DERIVATIVE_HISTORY_PATTERN = re.compile(
    r'^(?:Room_dTdt_central_|Room_d2Tdt2_central_|'
    r'ThermalSource_dTdt_central_|ThermalSource_d2Tdt2_central_)'
)

NO_DERIV_HISTORY_INPUTS = [
    c for c in FULL_MODEL_INPUTS
    if not DERIVATIVE_HISTORY_PATTERN.match(c)
]

removed_derivative_features = [
    c for c in FULL_MODEL_INPUTS
    if c not in NO_DERIV_HISTORY_INPUTS
]

if not removed_derivative_features:
    raise RuntimeError(
        'No derivative-history features were removed; ablation is invalid.'
    )

required_current = {
    'Room_T',
    'Room_dTdt_now_bwd2_C_per_min',
    'Room_d2Tdt2_now_bwd2_C_per_min2',
    'horizon_min',
    'thermal_mode_sign',
}
if not required_current.issubset(NO_DERIV_HISTORY_INPUTS):
    raise RuntimeError(
        'No-derivative-history ablation accidentally removed required '
        'current-state inputs.'
    )

input_audit = pd.DataFrame(audit)
input_audit.to_csv(
    AUDIT_DIR / 'NB05_full_model_input_reconstruction_audit.csv',
    index=False
)
pd.DataFrame({
    'feature': FULL_MODEL_INPUTS,
    'canonical_full': True,
    'no_derivative_history': [
        c in NO_DERIV_HISTORY_INPUTS for c in FULL_MODEL_INPUTS
    ],
}).to_csv(
    AUDIT_DIR / 'NB05_ablation_feature_sets.csv',
    index=False
)

print('Canonical input count:', len(FULL_MODEL_INPUTS))
print('No-derivative-history input count:', len(NO_DERIV_HISTORY_INPUTS))
print('Removed derivative-history features:', len(removed_derivative_features))
display(pd.DataFrame({'removed_feature': removed_derivative_features}))


## 3. Prepare transformations and loss parameters

Use the saved reference preprocessor for variants with the full feature set. For the reduced feature set, refit operating-mode medians and the combined scaler using training rows only. Retain the reference target scales and smooth completion-time parameters because the supervised targets are unchanged.


In [ ]:


# Exact reference preprocessor + reduced-feature preprocessor

def build_train_only_preprocessor(model_inputs):
    phase_medians = {}

    for phase in PHASES:
        z = (
            train_all.loc[
                train_all['phase'].eq(phase),
                model_inputs
            ]
            .apply(pd.to_numeric, errors='coerce')
            .replace([np.inf, -np.inf], np.nan)
        )

        phase_medians[phase] = {
            c: float(z[c].median())
            for c in model_inputs
        }

        bad = [
            c for c, v in phase_medians[phase].items()
            if not np.isfinite(v)
        ]
        if bad:
            raise RuntimeError(
                f'{phase}: no finite TRAIN median for {bad}'
            )

    def fill(df):
        x = (
            df[model_inputs]
            .apply(pd.to_numeric, errors='coerce')
            .replace([np.inf, -np.inf], np.nan)
            .copy()
        )
        phase_values = df['phase'].astype(str).to_numpy()

        for phase in PHASES:
            idx = np.where(phase_values == phase)[0]
            if len(idx) == 0:
                continue
            for c in model_inputs:
                col = x.columns.get_loc(c)
                vals = x.iloc[idx, col]
                x.iloc[idx, col] = vals.fillna(
                    phase_medians[phase][c]
                ).to_numpy()

        if x.isna().any().any():
            raise RuntimeError('TRAIN-only imputation left NaN values.')

        return x.to_numpy(dtype=float)

    scaler = StandardScaler().fit(fill(train_all))

    return {
        'numeric_cols': list(model_inputs),
        'phase_medians': phase_medians,
        'scaler': scaler,
        'include_room_identity': False,
        'imputation': 'phase_specific_LONG_train_median',
    }

def transform_with_preprocessor(df, prep):
    cols = list(prep['numeric_cols'])
    x = (
        df[cols]
        .apply(pd.to_numeric, errors='coerce')
        .replace([np.inf, -np.inf], np.nan)
        .copy()
    )

    phase_values = df['phase'].astype(str).to_numpy()
    for phase in PHASES:
        idx = np.where(phase_values == phase)[0]
        if len(idx) == 0:
            continue
        for c in cols:
            col = x.columns.get_loc(c)
            vals = x.iloc[idx, col]
            x.iloc[idx, col] = vals.fillna(
                float(prep['phase_medians'][phase][c])
            ).to_numpy()

    if x.isna().any().any():
        raise RuntimeError('Preprocessor left NaN values.')

    return prep['scaler'].transform(
        x.to_numpy(dtype=float)
    ).astype(np.float32)

# Reconstruct the reference preprocessor independently and require exact agreement.
REBUILT_CANONICAL_PREP = build_train_only_preprocessor(
    FULL_MODEL_INPUTS
)

for phase in PHASES:
    for c in FULL_MODEL_INPUTS:
        a = float(CANONICAL_PREPROCESSOR['phase_medians'][phase][c])
        b = float(REBUILT_CANONICAL_PREP['phase_medians'][phase][c])
        if not np.isclose(a, b, rtol=0.0, atol=1e-12):
            raise RuntimeError(
                f'Canonical phase median mismatch for {phase}/{c}: {a} vs {b}'
            )

if not np.allclose(
    CANONICAL_PREPROCESSOR['scaler'].mean_,
    REBUILT_CANONICAL_PREP['scaler'].mean_,
    rtol=0.0, atol=1e-12
):
    raise RuntimeError('Canonical StandardScaler means were not reproduced.')

if not np.allclose(
    CANONICAL_PREPROCESSOR['scaler'].scale_,
    REBUILT_CANONICAL_PREP['scaler'].scale_,
    rtol=0.0, atol=1e-12
):
    raise RuntimeError('Canonical StandardScaler scales were not reproduced.')

NO_DERIV_HISTORY_PREPROCESSOR = build_train_only_preprocessor(
    NO_DERIV_HISTORY_INPUTS
)
joblib.dump(
    NO_DERIV_HISTORY_PREPROCESSOR,
    COMMON_DIR / 'no_derivative_history_preprocessor.joblib'
)

with open(
    CANONICAL_MODEL_DIR / 'loss_scales_train_only_by_phase.json',
    'r',
    encoding='utf-8'
) as f:
    LOSS_SCALES = json.load(f)

with open(
    CANONICAL_MODEL_DIR / 'soft_crossing_parameters.json',
    'r',
    encoding='utf-8'
) as f:
    SOFT_CROSSING = json.load(f)

SOFT_TEMP_BETA_C = float(SOFT_CROSSING['temp_beta_C'])
SOFT_DIRECTION_BETA_BY_PHASE = {
    p: float(SOFT_CROSSING['direction_beta_C_by_phase'][p])
    for p in PHASES
}

PREPROCESSORS = {
    'full': CANONICAL_PREPROCESSOR,
    'no_derivative_history': NO_DERIV_HISTORY_PREPROCESSOR,
}

print('Canonical preprocessing reproduction: PASSED')
print('Loss scales:')
display(pd.DataFrame(LOSS_SCALES).T)
print('Soft crossing parameters:', SOFT_CROSSING)


## 4. Construct supervision datasets

Future trajectories provide rate and integrated-temperature targets; dedicated prediction rows provide remaining-duration targets. Build corresponding datasets for the full and reduced input sets, and verify duration-row coverage.


In [ ]:


# Supervision datasets — exact reference construction

class TemperatureTrajectoryDataset(Dataset):
    def __init__(self, frame, Xall):
        self.items = []
        d = frame[frame['use_for_temperature_training']].copy()

        for (phase, event_key, timestamp), g in d.groupby(
            ['phase', 'event_key', 'timestamp'],
            sort=False
        ):
            g = (
                g.sort_values('horizon_min')
                .drop_duplicates('horizon_min', keep='first')
            )

            h = pd.to_numeric(
                g['horizon_min'], errors='coerce'
            ).to_numpy(float)

            valid = np.isfinite(h) & (h > 0)
            g = g.iloc[np.where(valid)[0]].copy()
            h = h[valid]

            if len(g) == 0:
                continue

            order = np.argsort(h)
            g = g.iloc[order].copy()
            h = h[order]

            if np.any(
                np.abs(h / STEP_MIN - np.round(h / STEP_MIN)) > 1e-8
            ):
                continue

            rid = g['_row_id'].to_numpy(int)
            target_rate = pd.to_numeric(
                g['target_directed_rate_C_per_min'],
                errors='coerce'
            ).to_numpy(float)
            target_T = pd.to_numeric(
                g['target_Room_T'],
                errors='coerce'
            ).to_numpy(float)

            rate_mask = np.isfinite(target_rate)
            temp_mask = np.isfinite(target_T)

            if not temp_mask.any() or not rate_mask.any():
                continue

            first = g.iloc[0]
            v0_phys = pd.to_numeric(
                first.get(
                    'Room_dTdt_now_bwd2_C_per_min',
                    np.nan
                ),
                errors='coerce'
            )
            if not np.isfinite(v0_phys):
                past = pd.to_numeric(
                    first.get(
                        'Room_dTdt_central_-5min_C_per_min',
                        np.nan
                    ),
                    errors='coerce'
                )
                v0_phys = float(past) if np.isfinite(past) else 0.0

            self.items.append({
                'phase': str(phase),
                'event_key': str(event_key),
                'timestamp': pd.Timestamp(timestamp),
                'row_ids': rid,
                'X': Xall[rid],
                'h': h.astype(np.float32),
                'target_rate': target_rate.astype(np.float32),
                'rate_mask': rate_mask,
                'target_T': target_T.astype(np.float32),
                'temp_mask': temp_mask,
                'T0': float(first['Room_T']),
                'sign': float(first['thermal_mode_sign']),
                'v0_phys': float(v0_phys),
            })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]

def horizon_grid(phase):
    return np.arange(
        STEP_MIN,
        DURATION_MAX_HORIZON_BY_PHASE[phase] + 1e-9,
        STEP_MIN,
        dtype=float
    )

def repeat_anchor_for_horizons(row, phase):
    h = horizon_grid(phase)
    syn = pd.DataFrame([row.to_dict()] * len(h))
    syn['horizon_min'] = h
    syn['target_timestamp'] = (
        pd.Timestamp(row['feature_source_timestamp']) +
        pd.to_timedelta(h, unit='m')
    )
    return h, syn

class DurationAnchorDataset(Dataset):
    def __init__(self, frame, prep):
        self.items = []

        for _, row in frame.iterrows():
            phase = str(row['phase'])
            true_rem = pd.to_numeric(
                row['true_remaining_duration_min'],
                errors='coerce'
            )
            T0 = pd.to_numeric(row['Room_T'], errors='coerce')
            sign = pd.to_numeric(
                row['thermal_mode_sign'],
                errors='coerce'
            )
            target = pd.to_numeric(
                row['Phase_Target_T_C'],
                errors='coerce'
            )
            gap = pd.to_numeric(
                row['asof_gap_min'],
                errors='coerce'
            )
            q = pd.to_numeric(
                row['remaining_trajectory_quality_score'],
                errors='coerce'
            )

            if not all(
                np.isfinite(v)
                for v in [true_rem, T0, sign, target, gap, q]
            ):
                continue

            h, syn = repeat_anchor_for_horizons(row, phase)

            try:
                X = transform_with_preprocessor(syn, prep)
            except Exception as e:
                raise RuntimeError(
                    f'Failed to build synthetic inputs for anchor '
                    f'{row.get("anchor_key", "?")}: {e}'
                ) from e

            v0 = pd.to_numeric(
                row.get(
                    'Room_dTdt_now_bwd2_C_per_min',
                    np.nan
                ),
                errors='coerce'
            )
            if not np.isfinite(v0):
                v0 = pd.to_numeric(
                    row.get(
                        'Room_dTdt_central_-5min_C_per_min',
                        0.0
                    ),
                    errors='coerce'
                )
                if not np.isfinite(v0):
                    v0 = 0.0

            self.items.append({
                'phase': phase,
                'anchor_key': str(row['anchor_key']),
                'event_key': str(row['event_key']),
                'anchor_kind': str(
                    row.get('anchor_kind', 'anchor')
                ),
                'X': X,
                'h': h.astype(np.float32),
                'T0': float(T0),
                'sign': float(sign),
                'v0_phys': float(v0),
                'phase_target': float(target),
                'true_remaining': float(true_rem),
                'asof_gap': float(gap),
                'q': float(np.clip(q, 0, 1)),
            })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]

DATASETS = {}

for feature_set_name, prep in PREPROCESSORS.items():
    Xall = transform_with_preprocessor(DATA, prep)
    temp_sets = {}
    dur_sets = {}

    for split_name in ['train', 'valid', 'test']:
        for phase in PHASES:
            temp_sets[(split_name, phase)] = TemperatureTrajectoryDataset(
                DATA[
                    DATA['split'].eq(split_name) &
                    DATA['phase'].eq(phase)
                ],
                Xall
            )
            dur_sets[(split_name, phase)] = DurationAnchorDataset(
                ANCHORS[
                    ANCHORS['split'].eq(split_name) &
                    ANCHORS['phase'].eq(phase)
                ],
                prep
            )

    DATASETS[feature_set_name] = {
        'preprocessor': prep,
        'Xall': Xall,
        'temperature': temp_sets,
        'duration': dur_sets,
    }

# Coverage audit must equal the saved anchor universe for both feature sets.
coverage_rows = []

for feature_set_name, bundle in DATASETS.items():
    for split_name in ['train', 'valid', 'test']:
        for phase in PHASES:
            expected = int(
                ANCHORS[
                    ANCHORS['split'].eq(split_name) &
                    ANCHORS['phase'].eq(phase)
                ]['anchor_key'].nunique()
            )
            actual = len(
                bundle['duration'][(split_name, phase)]
            )
            coverage_rows.append({
                'feature_set': feature_set_name,
                'phase': phase,
                'split': split_name,
                'authoritative_anchor_count': expected,
                'duration_dataset_count': actual,
                'difference': actual - expected,
            })

COVERAGE_AUDIT = pd.DataFrame(coverage_rows)
COVERAGE_AUDIT.to_csv(
    AUDIT_DIR / 'NB05_authoritative_duration_supervision_coverage.csv',
    index=False
)

if (COVERAGE_AUDIT['difference'] != 0).any():
    raise RuntimeError(
        'Ablation dataset construction lost authoritative duration anchors.'
    )

display(COVERAGE_AUDIT)
print('All ablation datasets preserve authoritative duration-anchor coverage.')


## 5. Define the model and configurable loss

Reuse the reference network, rate integration and smooth completion-time calculation. Variant settings control duration weighting and the duration-loss coefficient.

For `No_duration_loss`, the duration forward pass still runs with a zero loss coefficient. This removes its training gradient while preserving the sequence of forward calls and dropout draws as closely as possible. Validation continues to use duration error.


In [ ]:


# Neural model, rate integration and configurable loss

class UnifiedDirectedRateNN(nn.Module):
    def __init__(
        self,
        n_in,
        hidden=(128, 64, 32),
        dropout=0.10
    ):
        super().__init__()
        layers = []
        prev = n_in

        for h in hidden:
            layers += [nn.Linear(prev, h), nn.Tanh()]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h

        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)

def _model_forward_fp32(model, x):
    with torch.autocast(
        device_type=DEVICE.type,
        dtype=AMP_DTYPE,
        enabled=USE_AMP
    ):
        y = model(x)
    return y.float()

def make_batches(ds, batch_size, shuffle=True, seed=0):
    idx = np.arange(len(ds))
    if shuffle:
        np.random.default_rng(seed).shuffle(idx)

    return [
        [ds.items[j] for j in idx[i:i + batch_size]]
        for i in range(0, len(idx), batch_size)
    ]

def stack_trajectory_batch(batch):
    B = len(batch)
    H = max(len(x['h']) for x in batch)
    F = batch[0]['X'].shape[1]

    X = np.zeros((B, H, F), np.float32)
    h = np.zeros((B, H), np.float32)
    target_rate = np.zeros((B, H), np.float32)
    target_T = np.zeros((B, H), np.float32)
    valid = np.zeros((B, H), bool)
    rmask = np.zeros((B, H), bool)
    tmask = np.zeros((B, H), bool)

    row_ids = []
    T0 = np.zeros(B, np.float32)
    sign = np.zeros(B, np.float32)
    v0 = np.zeros(B, np.float32)

    for b, item in enumerate(batch):
        n = len(item['h'])
        X[b, :n] = item['X']
        h[b, :n] = item['h']
        valid[b, :n] = True

        if n < H:
            h[b, n:] = item['h'][-1]

        target_rate[b, :n] = item['target_rate']
        target_T[b, :n] = item['target_T']
        rmask[b, :n] = item['rate_mask']
        tmask[b, :n] = item['temp_mask']

        T0[b] = item['T0']
        sign[b] = item['sign']
        v0[b] = item['v0_phys']
        row_ids.append(item['row_ids'])

    def tt(a, dtype=torch.float32):
        return torch.as_tensor(
            a,
            dtype=dtype,
            device=DEVICE
        )

    return {
        'X': tt(X),
        'h': tt(h),
        'target_rate': tt(target_rate),
        'target_T': tt(target_T),
        'valid': tt(valid, torch.bool),
        'rmask': tt(rmask, torch.bool),
        'tmask': tt(tmask, torch.bool),
        'T0': tt(T0),
        'sign': tt(sign),
        'v0': tt(v0),
        'row_ids': row_ids,
    }

def integrate_variable_batch_torch(
    h, pred_directed, T0, sign, v0
):
    pred_phys = sign[:, None] * pred_directed.float()
    dt = torch.cat(
        [h[:, :1], h[:, 1:] - h[:, :-1]],
        dim=1
    ).float()
    prev_rate = torch.cat(
        [v0[:, None], pred_phys[:, :-1]],
        dim=1
    )
    increments = 0.5 * dt * (prev_rate + pred_phys)

    return (
        T0[:, None] +
        torch.cumsum(increments, dim=1)
    )

def trajectory_batch_forward(model, batch):
    z = stack_trajectory_batch(batch)
    B, H, F = z['X'].shape

    pred = _model_forward_fp32(
        model,
        z['X'].reshape(B * H, F)
    ).reshape(B, H)

    pred_T = integrate_variable_batch_torch(
        z['h'],
        pred,
        z['T0'],
        z['sign'],
        z['v0']
    )

    return z, pred, pred_T

def trajectory_batch_loss(model, batch, phase):
    z, pred, pred_T = trajectory_batch_forward(
        model,
        batch
    )

    rs = float(LOSS_SCALES[phase]['rate_C_per_min'])
    ts = float(LOSS_SCALES[phase]['temperature_C'])

    rm = z['rmask'] & z['valid']
    tm = z['tmask'] & z['valid']

    if not rm.any() or not tm.any():
        raise RuntimeError(
            f'{phase}: empty trajectory supervision mask.'
        )

    Lr = torch.mean(
        ((pred[rm] - z['target_rate'][rm]) / rs) ** 2
    )
    Lt = torch.mean(
        ((pred_T[tm] - z['target_T'][tm]) / ts) ** 2
    )

    return {
        'trajectory': 0.5 * (Lr + Lt),
        'rate': Lr,
        'temperature': Lt,
    }

def stack_duration_batch(batch):
    X = torch.as_tensor(
        np.stack([x['X'] for x in batch]).astype(np.float32),
        device=DEVICE
    )
    h = torch.as_tensor(
        batch[0]['h'],
        dtype=torch.float32,
        device=DEVICE
    )

    if any(
        not np.array_equal(x['h'], batch[0]['h'])
        for x in batch
    ):
        raise RuntimeError('Duration batch horizon grids differ.')

    def tv(k):
        return torch.as_tensor(
            [x[k] for x in batch],
            dtype=torch.float32,
            device=DEVICE
        )

    return {
        'X': X,
        'h': h,
        'T0': tv('T0'),
        'sign': tv('sign'),
        'v0': tv('v0_phys'),
        'target': tv('phase_target'),
        'true': tv('true_remaining'),
        'gap': tv('asof_gap'),
        'q': tv('q'),
    }

def integrate_fixed_batch_torch(
    h, pred_directed, T0, sign, v0
):
    pred_phys = sign[:, None] * pred_directed.float()
    dt = torch.cat(
        [h[:1], h[1:] - h[:-1]]
    ).float()
    prev_rate = torch.cat(
        [v0[:, None], pred_phys[:, :-1]],
        dim=1
    )

    return (
        T0[:, None] +
        torch.cumsum(
            0.5 * dt[None, :] *
            (prev_rate + pred_phys),
            dim=1
        )
    )

def soft_crossing_batch_torch(
    h, pred_T, T0, target, sign, v0, phase
):
    threshold = target - sign * END_BAND_C
    temp_beta = float(SOFT_TEMP_BETA_C)
    dir_beta = float(
        SOFT_DIRECTION_BETA_BY_PHASE[phase]
    )

    a0 = (
        torch.sigmoid(
            sign * (T0 - threshold) / temp_beta
        ) *
        torch.sigmoid(
            sign * v0 * STEP_MIN / dir_beta
        )
    )

    prev_T = torch.cat(
        [T0[:, None], pred_T[:, :-1]],
        dim=1
    )

    a = (
        torch.sigmoid(
            sign[:, None] *
            (pred_T - threshold[:, None]) /
            temp_beta
        ) *
        torch.sigmoid(
            sign[:, None] *
            (pred_T - prev_T) /
            dir_beta
        )
    )

    one_minus = torch.clamp(
        1.0 - a,
        min=1e-7,
        max=1.0
    )
    prefix = torch.cumprod(one_minus, dim=1)

    survival_before = torch.cat(
        [
            (1 - a0)[:, None],
            (1 - a0)[:, None] * prefix[:, :-1]
        ],
        dim=1
    )

    p_first = survival_before * a
    survival_after = (1 - a0) * prefix[:, -1]

    return (
        torch.sum(
            p_first * h[None, :],
            dim=1
        ) +
        survival_after * h[-1]
    )

def duration_batch_forward(model, batch, phase):
    z = stack_duration_batch(batch)
    B, H, F = z['X'].shape

    pred = _model_forward_fp32(
        model,
        z['X'].reshape(B * H, F)
    ).reshape(B, H)

    pred_T = integrate_fixed_batch_torch(
        z['h'],
        pred,
        z['T0'],
        z['sign'],
        z['v0']
    )

    pred_source = soft_crossing_batch_torch(
        z['h'],
        pred_T,
        z['T0'],
        z['target'],
        z['sign'],
        z['v0'],
        phase
    )

    pred_anchor = torch.clamp(
        pred_source - z['gap'],
        min=0.0
    )

    return z, pred_anchor, pred_T, pred

def duration_batch_loss(
    model,
    batch,
    phase,
    q_mode='canonical'
):
    z, pred, _, _ = duration_batch_forward(
        model,
        batch,
        phase
    )

    ds = float(
        LOSS_SCALES[phase]['remaining_min']
    )
    sq = ((pred - z['true']) / ds) ** 2

    if q_mode == 'canonical':
        w = z['q']
    elif q_mode == 'uniform':
        w = torch.ones_like(z['q'])
    else:
        raise ValueError(
            f'Unknown q_mode: {q_mode}'
        )

    return (
        torch.sum(w * sq) /
        (torch.sum(w) + 1e-8)
    )

def evaluate_trajectory(
    model,
    ds,
    phase,
    batch_size=512
):
    model.eval()
    sums = {
        'trajectory': 0.0,
        'rate': 0.0,
        'temperature': 0.0
    }
    weight = 0

    with torch.inference_mode():
        for b in make_batches(
            ds, batch_size, False
        ):
            z = trajectory_batch_loss(
                model, b, phase
            )
            n = len(b)
            weight += n
            for k in sums:
                sums[k] += (
                    float(z[k].detach().cpu()) * n
                )

    return {
        k: v / max(weight, 1)
        for k, v in sums.items()
    }

def predict_duration_soft_dataset(
    model,
    ds,
    phase,
    batch_size=1024
):
    model.eval()
    rows = []

    with torch.inference_mode():
        for batch in make_batches(
            ds, batch_size, False
        ):
            z, pred, _, _ = duration_batch_forward(
                model, batch, phase
            )
            pv = pred.cpu().numpy()

            for item, p in zip(batch, pv):
                rows.append({
                    'anchor_key': item['anchor_key'],
                    'event_key': item['event_key'],
                    'anchor_kind': item['anchor_kind'],
                    'phase': phase,
                    'true_remaining_duration_min': item['true_remaining'],
                    'pred_remaining_duration_min': float(p),
                    'q': item['q'],
                })

    return pd.DataFrame(rows)

def duration_eval_metrics(model, ds, phase):
    d = predict_duration_soft_dataset(
        model, ds, phase
    )

    e = (
        d['pred_remaining_duration_min'].to_numpy(float) -
        d['true_remaining_duration_min'].to_numpy(float)
    )
    denom = np.maximum(
        d['true_remaining_duration_min'].to_numpy(float),
        STEP_MIN
    )
    dscl = float(
        LOSS_SCALES[phase]['remaining_min']
    )

    return {
        'duration_norm_mse_unweighted': float(
            np.mean((e / dscl) ** 2)
        ),
        'duration_MAE_min_unweighted': float(
            np.mean(np.abs(e))
        ),
        'duration_RMSE_min_unweighted': float(
            np.sqrt(np.mean(e ** 2))
        ),
        'duration_Bias_min_unweighted': float(
            np.mean(e)
        ),
        'duration_Median_RAE_pct_unweighted': float(
            100 * np.median(
                np.abs(e) / denom
            )
        ),
    }, d

# Forward/backward checks on both feature sets and both modes.
for feature_set_name, bundle in DATASETS.items():
    n_in = len(
        bundle['preprocessor']['numeric_cols']
    )
    _test_model = UnifiedDirectedRateNN(
        n_in,
        NN_HIDDEN_DIMS,
        0.0
    ).to(DEVICE)

    for phase in PHASES:
        _b = bundle['duration'][('train', phase)].items[
            :min(
                8,
                len(bundle['duration'][('train', phase)].items)
            )
        ]
        _loss = duration_batch_loss(
            _test_model,
            _b,
            phase,
            q_mode='canonical'
        )
        if not torch.isfinite(_loss):
            raise RuntimeError(
                f'{feature_set_name}/{phase}: non-finite duration loss.'
            )

print('Neural/physical utility smoke tests: PASSED')


## 6. Specify the interventions

Export the variant definitions before training. Check each feature set, weighting rule, loss coefficient and sharing setting against the intended intervention.


In [ ]:


# Ablation specifications + integrity checks

ABLATION_SPECS = {
    'No_q_weighting': {
        'feature_set': 'full',
        'q_mode': 'uniform',
        'duration_loss_coefficient': 1.0,
        'shared_across_phases': True,
        'description': (
            'Canonical PAPER-FINAL model with q replaced by 1 '
            'only in the duration loss.'
        ),
    },
    'No_duration_loss': {
        'feature_set': 'full',
        'q_mode': 'canonical',
        'duration_loss_coefficient': 0.0,
        'shared_across_phases': True,
        'description': (
            'Canonical PAPER-FINAL model with zero gradient '
            'coefficient on duration loss; duration validation '
            'criterion remains unchanged.'
        ),
    },
    'No_derivative_history': {
        'feature_set': 'no_derivative_history',
        'q_mode': 'canonical',
        'duration_loss_coefficient': 1.0,
        'shared_across_phases': True,
        'description': (
            'Canonical PAPER-FINAL model without historical '
            'room/source derivative lag features.'
        ),
    },
    'Separate_phase_NNs': {
        'feature_set': 'full',
        'q_mode': 'canonical',
        'duration_loss_coefficient': 1.0,
        'shared_across_phases': False,
        'description': (
            'Separate heating and cooling NNs with the same '
            'architecture, data, losses, and optimization settings.'
        ),
    },
}

unknown = [
    v for v in VARIANTS_TO_RUN
    if v not in ABLATION_SPECS
]
if unknown:
    raise RuntimeError(
        f'Unknown requested variants: {unknown}'
    )

# Verify that reference loss weighting is retained.
for forbidden_name in [
    'horizon_frequency_weight',
    'remaining_support_weight',
    'inverse_frequency_weight'
]:
    if forbidden_name in globals():
        raise RuntimeError(
            f'Forbidden BALANCED_FINAL concept present: {forbidden_name}'
        )

spec_table = pd.DataFrame([
    {
        'variant': k,
        **v
    }
    for k, v in ABLATION_SPECS.items()
])
spec_table.to_csv(
    AUDIT_DIR / 'NB05_ablation_specifications.csv',
    index=False
)

display(spec_table)


## 7. Configure variant training

Shared-network variants use the reference training loop with their specified intervention. Separate-mode networks cycle their own batches to retain the reference loop's number of optimizer steps per epoch. Each mode selects its own validation checkpoint.

All checkpoint selection uses validation data. Test predictions are generated after training and selection are complete.


In [ ]:


# Resumable ablation training engine

def capture_rng_state():
    st = {
        'numpy': np.random.get_state(),
        'python': random.getstate(),
        'torch': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        st['cuda'] = torch.cuda.get_rng_state_all()
    return st

def restore_rng_state(st):
    if not st:
        return
    np.random.set_state(st['numpy'])
    random.setstate(st['python'])
    torch.set_rng_state(st['torch'])
    if torch.cuda.is_available() and 'cuda' in st:
        torch.cuda.set_rng_state_all(st['cuda'])

# Reference global number of steps per epoch.
FULL_TEMP = DATASETS['full']['temperature']
FULL_DUR = DATASETS['full']['duration']

_reference_batches = {
    p: {
        't': len(
            make_batches(
                FULL_TEMP[('train', p)],
                NN_BATCH_TRAJECTORIES_PER_PHASE,
                False
            )
        ),
        'd': len(
            make_batches(
                FULL_DUR[('train', p)],
                NN_BATCH_DURATION_ANCHORS_PER_PHASE,
                False
            )
        ),
    }
    for p in PHASES
}

CANONICAL_GLOBAL_STEPS_PER_EPOCH = max(
    *[_reference_batches[p]['t'] for p in PHASES],
    *[_reference_batches[p]['d'] for p in PHASES],
)

print(
    'Canonical global optimizer steps/epoch:',
    CANONICAL_GLOBAL_STEPS_PER_EPOCH
)
print('Reference batch counts:', _reference_batches)

def variant_paths(variant, phase_model='shared'):
    root = VARIANT_ROOT / variant
    model_dir = root / 'models'
    table_dir = root / 'tables'
    checkpoint_dir = root / 'checkpoints'

    for p in [model_dir, table_dir, checkpoint_dir]:
        p.mkdir(parents=True, exist_ok=True)

    suffix = phase_model

    return {
        'root': root,
        'model_dir': model_dir,
        'table_dir': table_dir,
        'checkpoint_dir': checkpoint_dir,
        'best': model_dir / f'{suffix}_best.pt',
        'resume': checkpoint_dir / f'{suffix}_resume_state.pt',
        'complete': checkpoint_dir / f'{suffix}_TRAINING_COMPLETE.json',
        'history': table_dir / f'{suffix}_training_history.csv',
    }

def maybe_reset_variant(variant):
    if variant in RESET_VARIANTS:
        root = VARIANT_ROOT / variant
        if root.exists():
            shutil.rmtree(root)
        print('Reset variant:', variant)

def save_best_checkpoint(
    path,
    model,
    epoch,
    val_monitor,
    variant,
    feature_set,
    phase_model,
    extra
):
    torch.save({
        'state_dict': {
            k: v.detach().cpu()
            for k, v in model.state_dict().items()
        },
        'n_inputs': len(
            PREPROCESSORS[feature_set]['numeric_cols']
        ),
        'feature_names': list(
            PREPROCESSORS[feature_set]['numeric_cols']
        ),
        'hidden_dims': NN_HIDDEN_DIMS,
        'dropout': NN_DROPOUT,
        'variant': variant,
        'feature_set': feature_set,
        'phase_model': phase_model,
        'best_epoch': int(epoch),
        'best_validation_duration_norm_mse': float(val_monitor),
        'shared_split_sha256': SHARED_SPLIT_SHA256,
        'extra': extra,
    }, path)

def train_shared_variant(variant, spec):
    feature_set = spec['feature_set']
    bundle = DATASETS[feature_set]
    temp_sets = bundle['temperature']
    dur_sets = bundle['duration']
    n_inputs = len(
        bundle['preprocessor']['numeric_cols']
    )

    paths = variant_paths(
        variant,
        phase_model='shared'
    )

    reset_seeds(RANDOM_SEED)

    model = UnifiedDirectedRateNN(
        n_inputs,
        NN_HIDDEN_DIMS,
        NN_DROPOUT
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=NN_LR,
        weight_decay=NN_WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=NN_SCHEDULER_FACTOR,
        patience=NN_SCHEDULER_PATIENCE_VALIDATIONS,
        min_lr=NN_MIN_LR
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=USE_AMP
    )

    start_epoch = 1
    best_val = np.inf
    best_epoch = 0
    stale_validations = 0
    history = []

    if RESUME_TRAINING and paths['resume'].exists():
        ck = safe_torch_load(
            paths['resume'],
            map_location=DEVICE
        )
        if ck.get('shared_split_sha256') != SHARED_SPLIT_SHA256:
            raise RuntimeError(
                f'{variant}: resume split mismatch.'
            )
        if ck.get('variant') != variant:
            raise RuntimeError(
                f'{variant}: wrong resume checkpoint.'
            )

        model.load_state_dict(
            ck['model_state_dict']
        )
        optimizer.load_state_dict(
            ck['optimizer_state_dict']
        )
        scheduler.load_state_dict(
            ck['scheduler_state_dict']
        )

        if ck.get('scaler_state_dict') and USE_AMP:
            scaler.load_state_dict(
                ck['scaler_state_dict']
            )

        start_epoch = int(ck['epoch']) + 1
        best_val = float(ck['best_val'])
        best_epoch = int(ck['best_epoch'])
        stale_validations = int(
            ck['stale_validations']
        )
        history = ck.get('history', [])
        restore_rng_state(ck.get('rng_state'))

        print(
            f'{variant}: resuming at epoch {start_epoch} | '
            f'best={best_val:.6f} @ {best_epoch}'
        )

    if (
        paths['complete'].exists() and
        paths['best'].exists() and
        not FORCE_CONTINUE_AFTER_EARLY_STOP
    ):
        ck = safe_torch_load(
            paths['best'],
            map_location=DEVICE
        )
        model.load_state_dict(ck['state_dict'])
        model.eval()
        return model, pd.DataFrame(history)

    q_mode = spec['q_mode']
    duration_coeff = float(
        spec['duration_loss_coefficient']
    )

    for epoch in range(
        start_epoch,
        NN_MAX_EPOCHS + 1
    ):
        t_epoch = time.time()
        model.train()

        tb = {
            p: make_batches(
                temp_sets[('train', p)],
                NN_BATCH_TRAJECTORIES_PER_PHASE,
                True,
                RANDOM_SEED + epoch +
                (0 if p == 'heating' else 10000)
            )
            for p in PHASES
        }

        db = {
            p: make_batches(
                dur_sets[('train', p)],
                NN_BATCH_DURATION_ANCHORS_PER_PHASE,
                True,
                RANDOM_SEED + 20000 + epoch +
                (0 if p == 'heating' else 10000)
            )
            for p in PHASES
        }

        nsteps = max(
            *[len(tb[p]) for p in PHASES],
            *[len(db[p]) for p in PHASES]
        )

        if nsteps != CANONICAL_GLOBAL_STEPS_PER_EPOCH:
            raise RuntimeError(
                f'{variant}: optimizer-step count changed '
                f'({nsteps} vs canonical '
                f'{CANONICAL_GLOBAL_STEPS_PER_EPOCH}).'
            )

        cyc_t = {
            p: cycle(tb[p]) for p in PHASES
        }
        cyc_d = {
            p: cycle(db[p]) for p in PHASES
        }

        acc = []

        for _ in range(nsteps):
            optimizer.zero_grad(set_to_none=True)

            phase_losses = {}

            for phase in PHASES:
                lt = trajectory_batch_loss(
                    model,
                    next(cyc_t[phase]),
                    phase
                )

                # Always execute the duration forward pass. For
                # No_duration_loss the gradient coefficient is zero.
                ld = duration_batch_loss(
                    model,
                    next(cyc_d[phase]),
                    phase,
                    q_mode=q_mode
                )

                total_phase = (
                    lt['trajectory'] +
                    duration_coeff * ld
                )

                phase_losses[phase] = {
                    'trajectory': lt['trajectory'],
                    'rate': lt['rate'],
                    'temperature': lt['temperature'],
                    'duration': ld,
                    'total': total_phase,
                }

            total = 0.5 * (
                phase_losses['heating']['total'] +
                phase_losses['cooling']['total']
            )

            scaler.scale(total).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                5.0
            )
            scaler.step(optimizer)
            scaler.update()

            acc.append({
                'total': float(
                    total.detach().cpu()
                ),
                'trajectory': float(
                    (
                        0.5 * (
                            phase_losses['heating']['trajectory'] +
                            phase_losses['cooling']['trajectory']
                        )
                    ).detach().cpu()
                ),
                'duration_raw': float(
                    (
                        0.5 * (
                            phase_losses['heating']['duration'] +
                            phase_losses['cooling']['duration']
                        )
                    ).detach().cpu()
                ),
                'duration_gradient_coefficient': duration_coeff,
            })

        tr = {
            k: float(
                np.mean([x[k] for x in acc])
            )
            for k in acc[0]
        }

        rec = {
            'variant': variant,
            'epoch': epoch,
            **{
                f'train_{k}': v
                for k, v in tr.items()
            },
            'lr': float(
                optimizer.param_groups[0]['lr']
            ),
            'epoch_seconds': float(
                time.time() - t_epoch
            ),
            'validation_performed': False,
        }

        do_val = (
            epoch == 1 or
            epoch % NN_VALIDATION_EVERY == 0 or
            epoch == NN_MAX_EPOCHS
        )

        if do_val:
            tval = time.time()

            vt = {
                p: evaluate_trajectory(
                    model,
                    temp_sets[('valid', p)],
                    p
                )
                for p in PHASES
            }

            vd = {}
            for p in PHASES:
                vd[p], _ = duration_eval_metrics(
                    model,
                    dur_sets[('valid', p)],
                    p
                )

            # Reference model-selection metric.
            val_duration_loss = 0.5 * (
                vd['heating']['duration_norm_mse_unweighted'] +
                vd['cooling']['duration_norm_mse_unweighted']
            )

            val_duration_mae = 0.5 * (
                vd['heating']['duration_MAE_min_unweighted'] +
                vd['cooling']['duration_MAE_min_unweighted']
            )

            val_rel = 0.5 * (
                vd['heating']['duration_Median_RAE_pct_unweighted'] +
                vd['cooling']['duration_Median_RAE_pct_unweighted']
            )

            val_traj = 0.5 * (
                vt['heating']['trajectory'] +
                vt['cooling']['trajectory']
            )

            scheduler.step(val_duration_loss)

            rec.update({
                'validation_performed': True,
                'validation_seconds': float(
                    time.time() - tval
                ),
                'val_duration_norm_mse_unweighted_balanced':
                    val_duration_loss,
                'val_duration_MAE_min_unweighted_balanced':
                    val_duration_mae,
                'val_duration_Median_RAE_pct_unweighted_balanced':
                    val_rel,
                'val_trajectory_balanced':
                    val_traj,
                'val_heating_MAE_min':
                    vd['heating']['duration_MAE_min_unweighted'],
                'val_cooling_MAE_min':
                    vd['cooling']['duration_MAE_min_unweighted'],
                'val_heating_Median_RAE_pct':
                    vd['heating']['duration_Median_RAE_pct_unweighted'],
                'val_cooling_Median_RAE_pct':
                    vd['cooling']['duration_Median_RAE_pct_unweighted'],
                'lr_after_scheduler': float(
                    optimizer.param_groups[0]['lr']
                ),
            })

            improved = (
                val_duration_loss <
                best_val - NN_MIN_DELTA
            )

            if improved:
                best_val = val_duration_loss
                best_epoch = epoch
                stale_validations = 0

                save_best_checkpoint(
                    paths['best'],
                    model,
                    epoch,
                    val_duration_loss,
                    variant,
                    feature_set,
                    'shared',
                    rec
                )
            else:
                stale_validations += 1

            print(
                f'{variant} | epoch={epoch:03d} '
                f'valDurLoss={val_duration_loss:.5f} '
                f'traj={val_traj:.4f} '
                f'MAE H/C='
                f'{vd["heating"]["duration_MAE_min_unweighted"]:.2f}/'
                f'{vd["cooling"]["duration_MAE_min_unweighted"]:.2f} '
                f'| lr={optimizer.param_groups[0]["lr"]:.2e} '
                f'| best={best_epoch}'
            )

        history.append(rec)
        pd.DataFrame(history).to_csv(
            paths['history'],
            index=False
        )

        torch.save({
            'variant': variant,
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict':
                scaler.state_dict() if USE_AMP else None,
            'best_val': best_val,
            'best_epoch': best_epoch,
            'stale_validations': stale_validations,
            'history': history,
            'rng_state': capture_rng_state(),
            'shared_split_sha256': SHARED_SPLIT_SHA256,
        }, paths['resume'])

        if (
            do_val and
            stale_validations >=
            NN_EARLY_STOP_PATIENCE_VALIDATIONS
        ):
            print(
                f'{variant}: early stopping after '
                f'{stale_validations} stale validation checkpoints.'
            )
            break

    if not paths['best'].exists():
        raise RuntimeError(
            f'{variant}: training finished without best checkpoint.'
        )

    with open(
        paths['complete'],
        'w',
        encoding='utf-8'
    ) as f:
        json.dump({
            'variant': variant,
            'best_epoch': best_epoch,
            'best_validation_duration_norm_mse': best_val,
            'selection_metric':
                'balanced unweighted normalized validation duration MSE',
            'test_used_for_selection': False,
        }, f, indent=2)

    best = safe_torch_load(
        paths['best'],
        map_location=DEVICE
    )
    model.load_state_dict(best['state_dict'])
    model.eval()

    return model, pd.DataFrame(history)

def train_separate_phase_model(
    variant,
    phase,
    spec
):
    feature_set = spec['feature_set']
    bundle = DATASETS[feature_set]
    temp_sets = bundle['temperature']
    dur_sets = bundle['duration']
    n_inputs = len(
        bundle['preprocessor']['numeric_cols']
    )

    paths = variant_paths(
        variant,
        phase_model=phase
    )

    # Same initialization seed for both separate phase models; only the
    # original mode-specific batch-shuffle offsets are retained below.
    reset_seeds(RANDOM_SEED)

    model = UnifiedDirectedRateNN(
        n_inputs,
        NN_HIDDEN_DIMS,
        NN_DROPOUT
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=NN_LR,
        weight_decay=NN_WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=NN_SCHEDULER_FACTOR,
        patience=NN_SCHEDULER_PATIENCE_VALIDATIONS,
        min_lr=NN_MIN_LR
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=USE_AMP
    )

    start_epoch = 1
    best_val = np.inf
    best_epoch = 0
    stale_validations = 0
    history = []

    if RESUME_TRAINING and paths['resume'].exists():
        ck = safe_torch_load(
            paths['resume'],
            map_location=DEVICE
        )
        if ck.get('shared_split_sha256') != SHARED_SPLIT_SHA256:
            raise RuntimeError(
                f'{variant}/{phase}: resume split mismatch.'
            )

        model.load_state_dict(
            ck['model_state_dict']
        )
        optimizer.load_state_dict(
            ck['optimizer_state_dict']
        )
        scheduler.load_state_dict(
            ck['scheduler_state_dict']
        )

        if ck.get('scaler_state_dict') and USE_AMP:
            scaler.load_state_dict(
                ck['scaler_state_dict']
            )

        start_epoch = int(ck['epoch']) + 1
        best_val = float(ck['best_val'])
        best_epoch = int(ck['best_epoch'])
        stale_validations = int(
            ck['stale_validations']
        )
        history = ck.get('history', [])
        restore_rng_state(ck.get('rng_state'))

        print(
            f'{variant}/{phase}: resuming at epoch {start_epoch}'
        )

    if (
        paths['complete'].exists() and
        paths['best'].exists() and
        not FORCE_CONTINUE_AFTER_EARLY_STOP
    ):
        ck = safe_torch_load(
            paths['best'],
            map_location=DEVICE
        )
        model.load_state_dict(ck['state_dict'])
        model.eval()
        return model, pd.DataFrame(history)

    for epoch in range(
        start_epoch,
        NN_MAX_EPOCHS + 1
    ):
        t_epoch = time.time()
        model.train()

        tb = make_batches(
            temp_sets[('train', phase)],
            NN_BATCH_TRAJECTORIES_PER_PHASE,
            True,
            RANDOM_SEED + epoch +
            (0 if phase == 'heating' else 10000)
        )

        db = make_batches(
            dur_sets[('train', phase)],
            NN_BATCH_DURATION_ANCHORS_PER_PHASE,
            True,
            RANDOM_SEED + 20000 + epoch +
            (0 if phase == 'heating' else 10000)
        )

        cyc_t = cycle(tb)
        cyc_d = cycle(db)
        acc = []

        # Same number of updates per epoch as each phase branch receives
        # inside the reference shared model.
        for _ in range(
            CANONICAL_GLOBAL_STEPS_PER_EPOCH
        ):
            optimizer.zero_grad(set_to_none=True)

            lt = trajectory_batch_loss(
                model,
                next(cyc_t),
                phase
            )
            ld = duration_batch_loss(
                model,
                next(cyc_d),
                phase,
                q_mode='canonical'
            )
            total = lt['trajectory'] + ld

            scaler.scale(total).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                5.0
            )
            scaler.step(optimizer)
            scaler.update()

            acc.append({
                'total': float(total.detach().cpu()),
                'trajectory': float(
                    lt['trajectory'].detach().cpu()
                ),
                'duration_raw': float(
                    ld.detach().cpu()
                ),
            })

        tr = {
            k: float(
                np.mean([x[k] for x in acc])
            )
            for k in acc[0]
        }

        rec = {
            'variant': variant,
            'phase_model': phase,
            'epoch': epoch,
            **{
                f'train_{k}': v
                for k, v in tr.items()
            },
            'lr': float(
                optimizer.param_groups[0]['lr']
            ),
            'epoch_seconds': float(
                time.time() - t_epoch
            ),
            'validation_performed': False,
        }

        do_val = (
            epoch == 1 or
            epoch % NN_VALIDATION_EVERY == 0 or
            epoch == NN_MAX_EPOCHS
        )

        if do_val:
            tval = time.time()

            vt = evaluate_trajectory(
                model,
                temp_sets[('valid', phase)],
                phase
            )

            vd, _ = duration_eval_metrics(
                model,
                dur_sets[('valid', phase)],
                phase
            )

            # Phase-specific analogue of the reference validation criterion.
            val_duration_loss = (
                vd['duration_norm_mse_unweighted']
            )

            scheduler.step(val_duration_loss)

            rec.update({
                'validation_performed': True,
                'validation_seconds': float(
                    time.time() - tval
                ),
                'val_duration_norm_mse_unweighted':
                    val_duration_loss,
                'val_duration_MAE_min_unweighted':
                    vd['duration_MAE_min_unweighted'],
                'val_duration_Median_RAE_pct_unweighted':
                    vd['duration_Median_RAE_pct_unweighted'],
                'val_trajectory':
                    vt['trajectory'],
                'lr_after_scheduler': float(
                    optimizer.param_groups[0]['lr']
                ),
            })

            improved = (
                val_duration_loss <
                best_val - NN_MIN_DELTA
            )

            if improved:
                best_val = val_duration_loss
                best_epoch = epoch
                stale_validations = 0

                save_best_checkpoint(
                    paths['best'],
                    model,
                    epoch,
                    val_duration_loss,
                    variant,
                    feature_set,
                    phase,
                    rec
                )
            else:
                stale_validations += 1

            print(
                f'{variant}/{phase} | epoch={epoch:03d} '
                f'valDurLoss={val_duration_loss:.5f} '
                f'MAE={vd["duration_MAE_min_unweighted"]:.2f} '
                f'| lr={optimizer.param_groups[0]["lr"]:.2e} '
                f'| best={best_epoch}'
            )

        history.append(rec)
        pd.DataFrame(history).to_csv(
            paths['history'],
            index=False
        )

        torch.save({
            'variant': variant,
            'phase_model': phase,
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict':
                scaler.state_dict() if USE_AMP else None,
            'best_val': best_val,
            'best_epoch': best_epoch,
            'stale_validations': stale_validations,
            'history': history,
            'rng_state': capture_rng_state(),
            'shared_split_sha256': SHARED_SPLIT_SHA256,
        }, paths['resume'])

        if (
            do_val and
            stale_validations >=
            NN_EARLY_STOP_PATIENCE_VALIDATIONS
        ):
            print(
                f'{variant}/{phase}: early stopping after '
                f'{stale_validations} stale validation checkpoints.'
            )
            break

    if not paths['best'].exists():
        raise RuntimeError(
            f'{variant}/{phase}: no best checkpoint.'
        )

    with open(
        paths['complete'],
        'w',
        encoding='utf-8'
    ) as f:
        json.dump({
            'variant': variant,
            'phase_model': phase,
            'best_epoch': best_epoch,
            'best_validation_duration_norm_mse': best_val,
            'selection_metric':
                f'{phase} unweighted normalized validation duration MSE',
            'test_used_for_selection': False,
            'optimizer_steps_per_epoch':
                CANONICAL_GLOBAL_STEPS_PER_EPOCH,
        }, f, indent=2)

    best = safe_torch_load(
        paths['best'],
        map_location=DEVICE
    )
    model.load_state_dict(best['state_dict'])
    model.eval()

    return model, pd.DataFrame(history)


## 8. Train or resume variants

Load the original checkpoint without retraining it. Train or resume the four variants independently, preserving their optimizer and scheduler states where available.


In [ ]:


# Load reference model and train/resume ablations

for required in [
    CANONICAL_BEST_PATH,
    CANONICAL_MANIFEST_PATH,
    NB03_ALL_PRED_PATH,
    NB03_MAIN_PATH,
    NB03_MANIFEST_PATH,
]:
    if not required.exists():
        raise FileNotFoundError(required)

with open(
    CANONICAL_MANIFEST_PATH,
    'r',
    encoding='utf-8'
) as f:
    CANONICAL_MANIFEST = json.load(f)

with open(
    NB03_MANIFEST_PATH,
    'r',
    encoding='utf-8'
) as f:
    NB03_MANIFEST = json.load(f)

if CANONICAL_MANIFEST.get('shared_split_sha256') != SHARED_SPLIT_SHA256:
    raise RuntimeError(
        'Canonical Notebook-02 model used a different split.'
    )
if NB03_MANIFEST.get('shared_split_sha256') != SHARED_SPLIT_SHA256:
    raise RuntimeError(
        'Notebook-03 evaluation used a different split.'
    )

canonical_ck = safe_torch_load(
    CANONICAL_BEST_PATH,
    map_location=DEVICE
)

if int(canonical_ck['n_inputs']) != len(FULL_MODEL_INPUTS):
    raise RuntimeError(
        'Canonical checkpoint input dimension mismatch.'
    )

CANONICAL_MODEL = UnifiedDirectedRateNN(
    len(FULL_MODEL_INPUTS),
    NN_HIDDEN_DIMS,
    NN_DROPOUT
).to(DEVICE)
CANONICAL_MODEL.load_state_dict(
    canonical_ck['state_dict']
)
CANONICAL_MODEL.eval()

MODELS = {
    'Canonical_full': {
        'shared': True,
        'feature_set': 'full',
        'models': {'shared': CANONICAL_MODEL},
    }
}

HISTORIES = {}

for variant in VARIANTS_TO_RUN:
    maybe_reset_variant(variant)
    spec = ABLATION_SPECS[variant]

    if spec['shared_across_phases']:
        model, history = train_shared_variant(
            variant,
            spec
        )
        MODELS[variant] = {
            'shared': True,
            'feature_set': spec['feature_set'],
            'models': {'shared': model},
        }
        HISTORIES[variant] = {
            'shared': history
        }
    else:
        phase_models = {}
        phase_histories = {}

        for phase in PHASES:
            model, history = train_separate_phase_model(
                variant,
                phase,
                spec
            )
            phase_models[phase] = model
            phase_histories[phase] = history

        MODELS[variant] = {
            'shared': False,
            'feature_set': spec['feature_set'],
            'models': phase_models,
        }
        HISTORIES[variant] = phase_histories

print('\nAvailable model variants:')
for k, v in MODELS.items():
    print(
        k,
        '| shared=', v['shared'],
        '| feature_set=', v['feature_set']
    )


## 9. Verify reference prediction reproduction

Apply Notebook 03's evaluation completion-time procedure to the reference checkpoint and compare the resulting durations with the saved predictions. This check precedes the variant comparison.


In [ ]:


# Exact Notebook-03 evaluation completion-time procedure

def crossed(phase, T, threshold):
    return (
        float(T) >= float(threshold)
        if phase == 'heating'
        else float(T) <= float(threshold)
    )

def direction_valid(phase, dT):
    return (
        np.isfinite(dT) and
        (
            (dT > 0)
            if phase == 'heating'
            else (dT < 0)
        )
    )

def linear_crossing(
    h0, T0, h1, T1, threshold
):
    if abs(T1 - T0) < 1e-12:
        return float(h1)

    a = np.clip(
        (threshold - T0) / (T1 - T0),
        0, 1
    )

    return float(
        h0 + a * (h1 - h0)
    )

def estimate_crossing_from_source(
    phase,
    current_T,
    threshold,
    horizons,
    pred_T,
    current_delta_prev=np.nan
):
    h = np.asarray(horizons, float)
    y = np.asarray(pred_T, float)

    m = np.isfinite(h) & np.isfinite(y)
    h = h[m]
    y = y[m]

    if len(h) == 0:
        return (
            np.nan,
            'no_valid_trajectory',
            np.nan
        )

    o = np.argsort(h)
    h = h[o]
    y = y[o]

    h0 = np.r_[0.0, h]
    y0 = np.r_[float(current_T), y]

    if (
        crossed(
            phase,
            current_T,
            threshold
        ) and
        direction_valid(
            phase,
            current_delta_prev
        )
    ):
        return (
            0.0,
            'label_matched_at_source',
            float(h[-1])
        )

    ph, pt = h0[0], y0[0]

    for hh, tt in zip(h0[1:], y0[1:]):
        d = tt - pt

        if (
            crossed(
                phase,
                tt,
                threshold
            ) and
            direction_valid(
                phase,
                d
            )
        ):
            if not crossed(
                phase,
                pt,
                threshold
            ):
                return (
                    linear_crossing(
                        ph, pt,
                        hh, tt,
                        threshold
                    ),
                    'label_matched_interpolated_crossing',
                    float(h[-1])
                )

            return (
                float(hh),
                'label_matched_direction_at_horizon',
                float(h[-1])
            )

        ph, pt = hh, tt

    n = min(
        EXTRAPOLATE_FINAL_N_POINTS,
        len(h0)
    )

    if n >= 3:
        slope, intercept = np.polyfit(
            h0[-n:],
            y0[-n:],
            1
        )

        good = (
            (
                phase == 'heating' and
                slope >
                EXTRAPOLATE_MIN_ABS_SLOPE_C_PER_MIN
            ) or
            (
                phase == 'cooling' and
                slope <
                -EXTRAPOLATE_MIN_ABS_SLOPE_C_PER_MIN
            )
        )

        if good:
            hp = (
                float(threshold) -
                intercept
            ) / slope

            cap = float(
                h0[-1] +
                EXTRAPOLATE_MAX_EXTRA_MIN_BY_PHASE[phase]
            )

            if (
                np.isfinite(hp) and
                hp > h0[-1]
            ):
                return (
                    float(min(hp, cap)),
                    (
                        'label_matched_extrapolated_final_trend'
                        if hp <= cap
                        else 'projected_beyond_cap_used_cap'
                    ),
                    float(h[-1])
                )

    return (
        float(h0[-1]),
        'no_label_match_used_last_horizon',
        float(h[-1])
    )

def source_to_anchor_remaining(
    source_crossing_min,
    asof_gap_min
):
    if not np.isfinite(source_crossing_min):
        return np.nan

    return float(
        max(
            0.0,
            float(source_crossing_min) -
            float(asof_gap_min)
        )
    )

assert abs(
    source_to_anchor_remaining(
        35.0, 3.0
    ) - 32.0
) < 1e-12

def initial_rate_array(df):
    if 'Room_dTdt_now_bwd2_C_per_min' in df.columns:
        v = pd.to_numeric(
            df['Room_dTdt_now_bwd2_C_per_min'],
            errors='coerce'
        ).to_numpy(float)
    else:
        v = np.full(len(df), np.nan)

    if 'Room_dTdt_central_-5min_C_per_min' in df.columns:
        fb = pd.to_numeric(
            df['Room_dTdt_central_-5min_C_per_min'],
            errors='coerce'
        ).to_numpy(float)
    else:
        fb = np.zeros(len(df), dtype=float)

    bad = ~np.isfinite(v)
    v[bad] = fb[bad]
    v[~np.isfinite(v)] = 0.0

    return v

def integrate_numpy_batch(
    h,
    pred_directed,
    T0,
    sign,
    v0_phys
):
    pred_directed = np.asarray(
        pred_directed,
        float
    )
    T0 = np.asarray(T0, float)
    sign = np.asarray(sign, float)
    v0_phys = np.asarray(v0_phys, float)

    pred_phys = sign[:, None] * pred_directed
    dt = np.r_[
        h[0],
        np.diff(h)
    ].astype(float)

    prev = np.concatenate(
        [
            v0_phys[:, None],
            pred_phys[:, :-1]
        ],
        axis=1
    )

    return (
        T0[:, None] +
        np.cumsum(
            0.5 * dt[None, :] *
            (prev + pred_phys),
            axis=1
        )
    )

def make_expanded_anchor_frame(chunk, h):
    H = len(h)
    expanded = (
        chunk.loc[
            chunk.index.repeat(H)
        ]
        .copy()
        .reset_index(drop=True)
    )

    expanded['horizon_min'] = np.tile(
        h,
        len(chunk)
    )

    expanded['target_timestamp'] = (
        pd.to_datetime(
            expanded['feature_source_timestamp']
        ) +
        pd.to_timedelta(
            expanded['horizon_min'],
            unit='m'
        )
    )

    return expanded

def model_for_phase(model_bundle, phase):
    if model_bundle['shared']:
        return model_bundle['models']['shared']
    return model_bundle['models'][phase]

def predict_variant_hard(
    variant_name,
    model_bundle,
    anchors,
    chunk_size=1024
):
    feature_set = model_bundle['feature_set']
    prep = PREPROCESSORS[feature_set]
    records = []

    for phase in PHASES:
        ap = (
            anchors[
                anchors['phase'].eq(phase)
            ]
            .copy()
            .reset_index(drop=True)
        )

        h = horizon_grid(phase)
        H = len(h)
        model = model_for_phase(
            model_bundle,
            phase
        )
        model.eval()

        for lo in range(
            0,
            len(ap),
            chunk_size
        ):
            chunk = (
                ap.iloc[
                    lo:lo + chunk_size
                ]
                .copy()
                .reset_index(drop=True)
            )

            B = len(chunk)
            expanded = make_expanded_anchor_frame(
                chunk,
                h
            )

            X = transform_with_preprocessor(
                expanded,
                prep
            )

            xt = torch.from_numpy(
                X
            ).to(
                DEVICE,
                dtype=torch.float32,
                non_blocking=True
            )

            with torch.inference_mode():
                pred_directed = (
                    model(xt)
                    .float()
                    .reshape(B, H)
                    .cpu()
                    .numpy()
                )

            del xt

            sign_arr = pd.to_numeric(
                chunk['thermal_mode_sign'],
                errors='coerce'
            ).to_numpy(float)

            T0_arr = pd.to_numeric(
                chunk['Room_T'],
                errors='coerce'
            ).to_numpy(float)

            v0_arr = initial_rate_array(chunk)

            pT_all = integrate_numpy_batch(
                h,
                pred_directed,
                T0_arr,
                sign_arr,
                v0_arr
            )

            for bi in range(B):
                row = chunk.iloc[bi]

                current_delta = (
                    float(v0_arr[bi]) *
                    STEP_MIN
                )

                threshold = float(
                    row['Duration_Crossing_Threshold_C']
                )

                cross, ctype, maxh = (
                    estimate_crossing_from_source(
                        phase,
                        float(T0_arr[bi]),
                        threshold,
                        h,
                        pT_all[bi],
                        current_delta
                    )
                )

                pred_anchor = (
                    source_to_anchor_remaining(
                        cross,
                        float(
                            row['asof_gap_min']
                        )
                    )
                )

                records.append({
                    'variant': variant_name,
                    'anchor_key':
                        row['anchor_key'],
                    'phase': phase,
                    'room': row['room'],
                    'event_key':
                        row['event_key'],
                    'split': row['split'],
                    'anchor_kind':
                        row.get(
                            'anchor_kind',
                            'anchor'
                        ),
                    'anchor_timestamp':
                        row['anchor_timestamp'],
                    'feature_source_timestamp':
                        row['feature_source_timestamp'],
                    'asof_gap_min':
                        row['asof_gap_min'],
                    'true_remaining_duration_min':
                        row['true_remaining_duration_min'],
                    'remaining_trajectory_quality_score':
                        row['remaining_trajectory_quality_score'],
                    'pred_remaining_duration_min':
                        pred_anchor,
                    'crossing_from_feature_source_min':
                        cross,
                    'crossing_type':
                        ctype,
                    'max_horizon_min':
                        maxh,
                })

    return pd.DataFrame(records)

EVAL_ANCHORS = (
    ANCHORS[
        ANCHORS['split'].eq('test')
    ]
    .copy()
    .reset_index(drop=True)
)

# Replay reference model with Notebook-03 converter.
CANONICAL_REPLAY = predict_variant_hard(
    'Canonical_full_replay',
    MODELS['Canonical_full'],
    EVAL_ANCHORS
)

NB03_ALL = pd.read_csv(
    NB03_ALL_PRED_PATH
)

NB03_CANONICAL = (
    NB03_ALL[
        (NB03_ALL['split'].eq('test')) &
        (NB03_ALL['method'].eq('Unified_Derivative_NN'))
    ][
        [
            'anchor_key',
            'pred_remaining_duration_min'
        ]
    ]
    .drop_duplicates('anchor_key')
    .rename(columns={
        'pred_remaining_duration_min':
            'pred_nb03'
    })
)

replay_check = CANONICAL_REPLAY[
    [
        'anchor_key',
        'pred_remaining_duration_min'
    ]
].merge(
    NB03_CANONICAL,
    on='anchor_key',
    how='outer',
    validate='one_to_one'
)

if replay_check.isna().any().any():
    raise RuntimeError(
        'Canonical replay and Notebook-03 anchor sets differ.'
    )

replay_check['abs_diff_min'] = np.abs(
    replay_check['pred_remaining_duration_min'] -
    replay_check['pred_nb03']
)

max_replay_diff = float(
    replay_check['abs_diff_min'].max()
)

print(
    'Canonical hard-prediction replay max abs difference [min]:',
    max_replay_diff
)

if (
    STRICT_CANONICAL_REPLAY and
    max_replay_diff > CANONICAL_REPLAY_TOL_MIN
):
    raise RuntimeError(
        'Canonical replay failed. Notebook 05 would not be '
        'strictly comparable with Notebook 03.'
    )

print('Canonical Notebook-03 replay: PASSED')


## 10. Generate test predictions

Evaluate each trained variant using the same completion-time procedure. After the reference reproduction check passes, retain Notebook 03's saved reference predictions in the comparison tables to avoid introducing inference-rounding differences.


In [ ]:


# Generate test duration predictions for all variants

variant_prediction_frames = []

# Exact saved Notebook-03 reference predictions.
canonical_saved = (
    NB03_ALL[
        (NB03_ALL['split'].eq('test')) &
        (NB03_ALL['method'].eq('Unified_Derivative_NN'))
    ]
    .copy()
)
canonical_saved['variant'] = 'Canonical_full'

keep_cols = [
    'variant', 'anchor_key', 'phase', 'room',
    'event_key', 'split', 'anchor_kind',
    'anchor_timestamp', 'feature_source_timestamp',
    'asof_gap_min', 'true_remaining_duration_min',
    'remaining_trajectory_quality_score',
    'pred_remaining_duration_min',
    'crossing_from_feature_source_min',
    'crossing_type', 'max_horizon_min'
]
keep_cols = [
    c for c in keep_cols
    if c in canonical_saved.columns
]
variant_prediction_frames.append(
    canonical_saved[keep_cols]
)

for variant in VARIANTS_TO_RUN:
    print('\nHard TEST inference:', variant)
    pred = predict_variant_hard(
        variant,
        MODELS[variant],
        EVAL_ANCHORS
    )

    if (
        pred['anchor_key'].nunique() !=
        EVAL_ANCHORS['anchor_key'].nunique()
    ):
        raise RuntimeError(
            f'{variant}: incomplete TEST anchor coverage.'
        )

    if pred[
        'pred_remaining_duration_min'
    ].isna().any():
        raise RuntimeError(
            f'{variant}: non-finite hard TEST prediction.'
        )

    variant_prediction_frames.append(pred)

ALL_VARIANT_PRED = pd.concat(
    variant_prediction_frames,
    ignore_index=True,
    sort=False
)

ALL_VARIANT_PRED.to_csv(
    PRED_DIR / 'NB05_all_variant_hard_duration_predictions.csv',
    index=False
)

print('\nFinite TEST predictions per variant/phase:')
display(
    ALL_VARIANT_PRED.groupby(
        ['variant', 'phase'],
        as_index=False
    ).agg(
        anchors=('anchor_key', 'nunique'),
        events=('event_key', 'nunique'),
        finite_predictions=(
            'pred_remaining_duration_min',
            lambda s: int(
                pd.to_numeric(
                    s, errors='coerce'
                ).notna().sum()
            )
        )
    )
)


## 11. Compare duration accuracy on common cases

Reconstruct the test prediction keys shared by the three main models in Notebook 03, then evaluate every neural variant on those keys. Export full-test neural results separately to distinguish coverage from the common-case comparison.


In [ ]:


# Exact-common support + Notebook-03 duration metrics

PAPER_METHODS = [
    'Direct_RoomPhase_ET',
    'Indirect_RoomPhase_ET',
    'Unified_Derivative_NN',
]

nb03_test = NB03_ALL[
    NB03_ALL['split'].eq('test')
].copy()

wide = nb03_test.pivot_table(
    index='anchor_key',
    columns='method',
    values='pred_remaining_duration_min',
    aggfunc='first'
)

for m in PAPER_METHODS:
    if m not in wide.columns:
        wide[m] = np.nan

EXACT_COMMON_KEYS = set(
    wide.dropna(
        subset=PAPER_METHODS
    ).index.astype(str)
)

def duration_metrics(g):
    true = pd.to_numeric(
        g['true_remaining_duration_min'],
        errors='coerce'
    ).to_numpy(float)

    pred = pd.to_numeric(
        g['pred_remaining_duration_min'],
        errors='coerce'
    ).to_numpy(float)

    good = np.isfinite(true) & np.isfinite(pred)
    true = true[good]
    pred = pred[good]

    e = pred - true
    ae = np.abs(e)
    denom = np.maximum(true, STEP_MIN)
    rae = 100.0 * ae / denom
    rse = 100.0 * e / denom

    return pd.Series({
        'n': len(e),
        'MAE_min': float(np.mean(ae)),
        'RMSE_min': float(
            np.sqrt(np.mean(e ** 2))
        ),
        'Bias_min': float(np.mean(e)),
        'MedianAE_min': float(
            np.median(ae)
        ),
        'P90AE_min': float(
            np.percentile(ae, 90)
        ),
        'MeanRAE_pct': float(
            np.mean(rae)
        ),
        'MedianRAE_pct': float(
            np.median(rae)
        ),
        'P90RAE_pct': float(
            np.percentile(rae, 90)
        ),
        'RelativeBias_pct': float(
            np.mean(rse)
        ),
    })

PRED_COMMON = ALL_VARIANT_PRED[
    ALL_VARIANT_PRED['anchor_key']
    .astype(str)
    .isin(EXACT_COMMON_KEYS)
].copy()

PRIMARY = (
    PRED_COMMON.groupby(
        ['phase', 'variant'],
        as_index=False
    )
    .apply(
        duration_metrics,
        include_groups=False
    )
    .reset_index(drop=True)
)

FULL_TEST = (
    ALL_VARIANT_PRED.groupby(
        ['phase', 'variant'],
        as_index=False
    )
    .apply(
        duration_metrics,
        include_groups=False
    )
    .reset_index(drop=True)
)

PRIMARY.to_csv(
    TABLE_DIR / 'PAPER_NB05_ablation_duration_metrics_EXACT_COMMON_SUPPORT.csv',
    index=False
)
FULL_TEST.to_csv(
    TABLE_DIR / 'PAPER_NB05_ablation_duration_metrics_FULL_TEST_SUPPORT.csv',
    index=False
)

# Deltas relative to the reference full Strategy III.
baseline = (
    PRIMARY[
        PRIMARY['variant'].eq(
            'Canonical_full'
        )
    ][
        [
            'phase',
            'MAE_min',
            'RMSE_min',
            'MedianAE_min',
            'MedianRAE_pct'
        ]
    ]
    .rename(columns={
        'MAE_min': 'Canonical_MAE_min',
        'RMSE_min': 'Canonical_RMSE_min',
        'MedianAE_min': 'Canonical_MedianAE_min',
        'MedianRAE_pct': 'Canonical_MedianRAE_pct',
    })
)

ABLATION_DELTA = PRIMARY.merge(
    baseline,
    on='phase',
    how='left',
    validate='many_to_one'
)

ABLATION_DELTA['Delta_MAE_vs_Canonical_min'] = (
    ABLATION_DELTA['MAE_min'] -
    ABLATION_DELTA['Canonical_MAE_min']
)
ABLATION_DELTA['Delta_RMSE_vs_Canonical_min'] = (
    ABLATION_DELTA['RMSE_min'] -
    ABLATION_DELTA['Canonical_RMSE_min']
)
ABLATION_DELTA['Delta_MedianAE_vs_Canonical_min'] = (
    ABLATION_DELTA['MedianAE_min'] -
    ABLATION_DELTA['Canonical_MedianAE_min']
)
ABLATION_DELTA['Delta_MedianRAE_vs_Canonical_pctpt'] = (
    ABLATION_DELTA['MedianRAE_pct'] -
    ABLATION_DELTA['Canonical_MedianRAE_pct']
)

ABLATION_DELTA.to_csv(
    TABLE_DIR / 'PAPER_NB05_ablation_deltas_vs_canonical.csv',
    index=False
)

print('PRIMARY ABLATION RESULTS — EXACT PAPER COMMON SUPPORT')
display(
    ABLATION_DELTA[
        [
            'phase', 'variant', 'n',
            'MAE_min', 'RMSE_min',
            'MedianAE_min', 'MedianRAE_pct',
            'Delta_MAE_vs_Canonical_min'
        ]
    ].round(3)
)


## 12. Compare temperature and rate accuracy

Evaluate each variant on the same test trajectory targets. These secondary measures help distinguish changes in average temperature/rate accuracy from changes in the inferred remaining duration.


In [ ]:


# TEST temperature/rate predictions for every NN variant

def predict_test_trajectory_variant(
    variant_name,
    model_bundle
):
    feature_set = model_bundle['feature_set']
    bundle = DATASETS[feature_set]
    rows = []

    for phase in PHASES:
        ds = bundle['temperature'][('test', phase)]
        model = model_for_phase(
            model_bundle,
            phase
        )
        model.eval()

        with torch.inference_mode():
            for batch in make_batches(
                ds,
                512,
                False
            ):
                z, pv, pT = trajectory_batch_forward(
                    model,
                    batch
                )

                pv = pv.cpu().numpy()
                pT = pT.cpu().numpy()
                valid = z['valid'].cpu().numpy()

                for b, item in enumerate(batch):
                    n = int(valid[b].sum())

                    for rid, v, t in zip(
                        item['row_ids'][:n],
                        pv[b, :n],
                        pT[b, :n]
                    ):
                        rows.append({
                            'variant': variant_name,
                            '_row_id': int(rid),
                            'phase': phase,
                            'pred_directed_rate': float(v),
                            'pred_T': float(t),
                        })

    return pd.DataFrame(rows)

traj_frames = []

for variant, model_bundle in MODELS.items():
    print('Trajectory TEST inference:', variant)
    traj_frames.append(
        predict_test_trajectory_variant(
            variant,
            model_bundle
        )
    )

TRAJ_PRED = pd.concat(
    traj_frames,
    ignore_index=True
)

truth_cols = [
    '_row_id', 'phase', 'room', 'event_key',
    'split', 'target_context', 'horizon_min',
    'target_Room_T',
    'target_directed_rate_C_per_min'
]

TRAJ_PRED = TRAJ_PRED.merge(
    DATA[truth_cols],
    on=['_row_id', 'phase'],
    how='left',
    validate='many_to_one'
)

TRAJ_PRED['T_error_C'] = (
    pd.to_numeric(
        TRAJ_PRED['pred_T'],
        errors='coerce'
    ) -
    pd.to_numeric(
        TRAJ_PRED['target_Room_T'],
        errors='coerce'
    )
)

TRAJ_PRED['rate_error_C_per_min'] = (
    pd.to_numeric(
        TRAJ_PRED['pred_directed_rate'],
        errors='coerce'
    ) -
    pd.to_numeric(
        TRAJ_PRED['target_directed_rate_C_per_min'],
        errors='coerce'
    )
)

TRAJ_PRED.to_csv(
    PRED_DIR / 'NB05_all_variant_TEST_trajectory_predictions.csv',
    index=False
)

def trajectory_metrics(g):
    te = pd.to_numeric(
        g['T_error_C'],
        errors='coerce'
    ).dropna().to_numpy(float)

    re = pd.to_numeric(
        g['rate_error_C_per_min'],
        errors='coerce'
    ).dropna().to_numpy(float)

    return pd.Series({
        'n_temperature_rows': len(te),
        'T_MAE_C': float(
            np.mean(np.abs(te))
        ),
        'T_RMSE_C': float(
            np.sqrt(np.mean(te ** 2))
        ),
        'T_Bias_C': float(np.mean(te)),
        'n_rate_rows': len(re),
        'Rate_MAE_C_per_min': float(
            np.mean(np.abs(re))
        ),
        'Rate_RMSE_C_per_min': float(
            np.sqrt(np.mean(re ** 2))
        ),
        'Rate_Bias_C_per_min': float(
            np.mean(re)
        ),
    })

TRAJ_METRICS = (
    TRAJ_PRED.groupby(
        ['phase', 'variant'],
        as_index=False
    )
    .apply(
        trajectory_metrics,
        include_groups=False
    )
    .reset_index(drop=True)
)

TRAJ_METRICS.to_csv(
    TABLE_DIR / 'PAPER_NB05_ablation_TEST_trajectory_metrics.csv',
    index=False
)

print('Secondary TEST trajectory metrics:')
display(
    TRAJ_METRICS.round(4)
)


## 13. Estimate paired uncertainty in duration differences

Resample whole test events with replacement, using identical sampled events for the reference model and each variant. Report

\[
\Delta\mathrm{MAE}=\mathrm{MAE}_{\mathrm{variant}}-\mathrm{MAE}_{\mathrm{reference}}.
\]

Positive differences indicate higher variant error. Bootstrap sign fractions are descriptive, not p-values.


### Prediction alignment

Reshape predictions using the unique `anchor_key`, then reattach operating mode, room, event and observed duration from the test metadata. This preserves one paired prediction row per key without using redundant metadata as pivot identifiers.


In [ ]:
# Align paired predictions by anchor_key before reattaching metadata.
def percentile_ci(x, ci_level=CI_LEVEL):
    a = np.asarray(pd.Series(x).dropna(), float)
    alpha = 1.0 - ci_level
    return (
        float(np.quantile(a, alpha / 2.0)),
        float(np.quantile(a, 1.0 - alpha / 2.0)),
    )

required_variants = ['Canonical_full'] + VARIANTS_TO_RUN
expected_keys = set(map(str, EXACT_COMMON_KEYS))

# Strict coverage audit before reshaping

coverage_audit_rows = []

for variant in required_variants:
    v = PRED_COMMON[PRED_COMMON['variant'].eq(variant)].copy()
    v['anchor_key'] = v['anchor_key'].astype(str)

    if v.duplicated(['anchor_key', 'variant']).any():
        dup = v.loc[
            v.duplicated(['anchor_key', 'variant'], keep=False),
            ['anchor_key', 'variant']
        ].head(20)
        raise RuntimeError(
            f'{variant}: duplicate anchor predictions detected.\n'
            + dup.to_string(index=False)
        )

    keys = set(v['anchor_key'])
    missing = sorted(expected_keys - keys)
    extra = sorted(keys - expected_keys)

    pred_num = pd.to_numeric(
        v['pred_remaining_duration_min'],
        errors='coerce'
    )
    nonfinite = int((~np.isfinite(pred_num.to_numpy(float))).sum())

    coverage_audit_rows.append({
        'variant': variant,
        'expected_exact_common_anchors': len(expected_keys),
        'present_anchors': len(keys),
        'missing_anchors': len(missing),
        'extra_anchors': len(extra),
        'nonfinite_predictions': nonfinite,
    })

    if missing or extra or nonfinite:
        raise RuntimeError(
            f'{variant}: exact-common coverage mismatch. '
            f'missing={len(missing)}, extra={len(extra)}, '
            f'nonfinite={nonfinite}. '
            f'First missing={missing[:5]}, first extra={extra[:5]}'
        )

BOOTSTRAP_COVERAGE_AUDIT = pd.DataFrame(coverage_audit_rows)
BOOTSTRAP_COVERAGE_AUDIT.to_csv(
    AUDIT_DIR / 'NB05_bootstrap_exact_common_variant_coverage_audit.csv',
    index=False
)

print('Bootstrap exact-common coverage audit:')
display(BOOTSTRAP_COVERAGE_AUDIT)

# Pivot ONLY on anchor_key.
# Redundant metadata are reattached from saved anchors.

pred_for_pivot = PRED_COMMON[
    ['anchor_key', 'variant', 'pred_remaining_duration_min']
].copy()
pred_for_pivot['anchor_key'] = pred_for_pivot['anchor_key'].astype(str)

wide_variant = (
    pred_for_pivot
    .pivot(
        index='anchor_key',
        columns='variant',
        values='pred_remaining_duration_min'
    )
    .reset_index()
)

missing_variant_columns = [
    v for v in required_variants
    if v not in wide_variant.columns
]
if missing_variant_columns:
    raise RuntimeError(
        f'Bootstrap pivot is missing variant columns: {missing_variant_columns}'
    )

if wide_variant[required_variants].isna().any().any():
    bad = wide_variant.loc[
        wide_variant[required_variants].isna().any(axis=1),
        ['anchor_key'] + required_variants
    ].head(20)
    raise RuntimeError(
        'Missing predictions remain after anchor_key-only pivot. '
        'This now indicates a real coverage problem.\n'
        + bad.to_string(index=False)
    )

# Reattach phase/event/truth metadata from saved TEST anchors

anchor_meta = EVAL_ANCHORS[
    [
        'anchor_key',
        'phase',
        'room',
        'event_key',
        'true_remaining_duration_min'
    ]
].copy()
anchor_meta['anchor_key'] = anchor_meta['anchor_key'].astype(str)

if anchor_meta['anchor_key'].duplicated().any():
    raise RuntimeError('Authoritative EVAL_ANCHORS has duplicate anchor_key.')

anchor_meta = anchor_meta[
    anchor_meta['anchor_key'].isin(expected_keys)
].copy()

if set(anchor_meta['anchor_key']) != expected_keys:
    missing_meta = sorted(
        expected_keys - set(anchor_meta['anchor_key'])
    )
    raise RuntimeError(
        f'Authoritative metadata missing exact-common anchors: {missing_meta[:10]}'
    )

wide_variant = anchor_meta.merge(
    wide_variant,
    on='anchor_key',
    how='inner',
    validate='one_to_one'
)

if len(wide_variant) != len(expected_keys):
    raise RuntimeError(
        f'Bootstrap table row count mismatch: '
        f'{len(wide_variant)} vs expected {len(expected_keys)}'
    )

check_cols = required_variants + ['true_remaining_duration_min']
if wide_variant[check_cols].isna().any().any():
    raise RuntimeError(
        'Bootstrap table contains missing values after authoritative metadata merge.'
    )

for c in check_cols:
    vals = pd.to_numeric(wide_variant[c], errors='coerce').to_numpy(float)
    if not np.isfinite(vals).all():
        raise RuntimeError(f'Bootstrap table contains non-finite values in {c}.')

print(
    'Bootstrap wide table constructed correctly:',
    len(wide_variant),
    'exact-common anchors.'
)

# Paired event-cluster bootstrap

rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_rows = []

for phase in PHASES:
    d = (
        wide_variant[wide_variant['phase'].eq(phase)]
        .copy()
        .reset_index(drop=True)
    )

    events = d['event_key'].astype(str).drop_duplicates().tolist()
    if not events:
        raise RuntimeError(f'{phase}: no exact-common events for bootstrap.')

    event_series = d['event_key'].astype(str)
    rows_by_event = {
        e: np.flatnonzero(event_series.eq(e).to_numpy())
        for e in events
    }

    true_all = pd.to_numeric(
        d['true_remaining_duration_min'],
        errors='coerce'
    ).to_numpy(float)

    pred_arrays = {
        variant: pd.to_numeric(
            d[variant],
            errors='coerce'
        ).to_numpy(float)
        for variant in required_variants
    }

    for b in range(N_BOOTSTRAP):
        sampled_events = rng.choice(
            events,
            size=len(events),
            replace=True
        )

        # Preserve event multiplicity.
        idx = np.concatenate([
            rows_by_event[e]
            for e in sampled_events
        ])

        true = true_all[idx]
        canonical = pred_arrays['Canonical_full'][idx]
        canonical_mae = float(
            np.mean(np.abs(canonical - true))
        )

        for variant in VARIANTS_TO_RUN:
            pred = pred_arrays[variant][idx]
            ablation_mae = float(
                np.mean(np.abs(pred - true))
            )

            bootstrap_rows.append({
                'phase': phase,
                'replicate': b,
                'variant': variant,
                'Canonical_MAE_min': canonical_mae,
                'Ablation_MAE_min': ablation_mae,
                'Delta_MAE_Ablation_minus_Canonical_min':
                    ablation_mae - canonical_mae,
            })

BOOT = pd.DataFrame(bootstrap_rows)

BOOT.to_csv(
    TABLE_DIR / 'NB05_ablation_event_cluster_bootstrap_replicates.csv.gz',
    index=False,
    compression='gzip'
)

summary_rows = []

for (phase, variant), g in BOOT.groupby(
    ['phase', 'variant'],
    sort=False
):
    delta = g[
        'Delta_MAE_Ablation_minus_Canonical_min'
    ].to_numpy(float)

    lo, hi = percentile_ci(delta)

    observed = float(
        ABLATION_DELTA.loc[
            (ABLATION_DELTA['phase'].eq(phase)) &
            (ABLATION_DELTA['variant'].eq(variant)),
            'Delta_MAE_vs_Canonical_min'
        ].iloc[0]
    )

    summary_rows.append({
        'phase': phase,
        'variant': variant,
        'observed_Delta_MAE_Ablation_minus_Canonical_min': observed,
        'CI_level': CI_LEVEL,
        'CI_low': lo,
        'CI_high': hi,
        'bootstrap_mean_delta': float(np.mean(delta)),
        'bootstrap_fraction_delta_gt_0': float(np.mean(delta > 0.0)),
        'bootstrap_fraction_delta_lt_0': float(np.mean(delta < 0.0)),
        'bootstrap_unit': 'physical event',
    })

ABLATION_BOOTSTRAP = pd.DataFrame(summary_rows)

ABLATION_BOOTSTRAP.to_csv(
    TABLE_DIR / 'PAPER_NB05_ablation_paired_event_cluster_bootstrap_CI.csv',
    index=False
)

print('Paired event-cluster bootstrap — ablation minus canonical MAE:')
display(ABLATION_BOOTSTRAP.round(3))


## 14. Summarize component comparisons

Combine duration errors with paired event-bootstrap intervals and plot the reference and variant results. Keep the direction of each difference explicit and report heating and cooling separately.


In [ ]:


# Paper-ready ablation table and figures

PAPER_ABLATION = (
    ABLATION_DELTA[
        ABLATION_DELTA['variant'].isin(
            VARIANTS_TO_RUN
        )
    ][
        [
            'phase', 'variant', 'n',
            'MAE_min', 'RMSE_min',
            'MedianAE_min', 'MedianRAE_pct',
            'Delta_MAE_vs_Canonical_min'
        ]
    ]
    .merge(
        ABLATION_BOOTSTRAP[
            [
                'phase', 'variant',
                'CI_low', 'CI_high',
                'bootstrap_fraction_delta_gt_0'
            ]
        ],
        on=['phase', 'variant'],
        how='left',
        validate='one_to_one'
    )
)

PAPER_ABLATION.to_csv(
    TABLE_DIR / 'PAPER_NB05_core_ablation_summary.csv',
    index=False
)

print('PAPER-READY CORE ABLATION SUMMARY')
display(
    PAPER_ABLATION.round(3)
)

# Figure 1 — primary MAE for reference + ablations.
plot_df = PRIMARY.copy()
variant_order = [
    'Canonical_full',
    'No_q_weighting',
    'No_duration_loss',
    'No_derivative_history',
    'Separate_phase_NNs'
]

fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 5),
    constrained_layout=True
)

for ax, phase in zip(axes, PHASES):
    d = (
        plot_df[
            plot_df['phase'].eq(phase)
        ]
        .set_index('variant')
        .reindex(variant_order)
        .reset_index()
    )

    x = np.arange(len(d))
    ax.bar(
        x,
        d['MAE_min']
    )
    ax.set_xticks(
        x,
        [
            'Canonical',
            'No q',
            'No duration\nloss',
            'No derivative\nhistory',
            'Separate\nphase NNs',
        ],
        rotation=15
    )
    ax.set_ylabel('MAE (min)')
    ax.set_title(
        f'{phase.capitalize()} — exact-common TEST'
    )
    ax.grid(
        axis='y',
        alpha=.25
    )

fig.savefig(
    FIG_DIR / 'PAPER_NB05_core_ablation_MAE.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

# Figure 2 — paired ΔMAE with event-cluster bootstrap CI.
fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 5),
    constrained_layout=True
)

for ax, phase in zip(axes, PHASES):
    d = (
        PAPER_ABLATION[
            PAPER_ABLATION['phase'].eq(phase)
        ]
        .set_index('variant')
        .reindex(VARIANTS_TO_RUN)
        .reset_index()
    )

    y = np.arange(len(d))
    center = d[
        'Delta_MAE_vs_Canonical_min'
    ].to_numpy(float)
    lo = d['CI_low'].to_numpy(float)
    hi = d['CI_high'].to_numpy(float)

    ax.errorbar(
        center,
        y,
        xerr=np.vstack([
            center - lo,
            hi - center
        ]),
        fmt='o',
        capsize=4
    )
    ax.axvline(
        0.0,
        linestyle='--'
    )
    ax.set_yticks(
        y,
        [
            'No q weighting',
            'No duration loss',
            'No derivative history',
            'Separate phase NNs',
        ]
    )
    ax.set_xlabel(
        'ΔMAE: ablation − canonical (min)'
    )
    ax.set_title(
        phase.capitalize()
    )
    ax.grid(
        axis='x',
        alpha=.25
    )

fig.savefig(
    FIG_DIR / 'PAPER_NB05_core_ablation_delta_MAE_cluster_bootstrap_CI.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()


## 15. Save provenance and integrity checks

Record the reference run, split fingerprint, intervention definitions, selected checkpoints and output files. Verify that the reference model and validation-based selection procedure remain consistent with the earlier notebooks.


In [ ]:


# Final Notebook-05 integrity manifest

best_checkpoint_records = []

for variant in VARIANTS_TO_RUN:
    spec = ABLATION_SPECS[variant]

    if spec['shared_across_phases']:
        p = variant_paths(
            variant,
            'shared'
        )
        ck = safe_torch_load(
            p['best'],
            map_location='cpu'
        )
        best_checkpoint_records.append({
            'variant': variant,
            'phase_model': 'shared',
            'best_epoch': int(
                ck['best_epoch']
            ),
            'best_validation_duration_norm_mse':
                float(
                    ck[
                        'best_validation_duration_norm_mse'
                    ]
                ),
            'checkpoint': str(p['best']),
        })
    else:
        for phase in PHASES:
            p = variant_paths(
                variant,
                phase
            )
            ck = safe_torch_load(
                p['best'],
                map_location='cpu'
            )
            best_checkpoint_records.append({
                'variant': variant,
                'phase_model': phase,
                'best_epoch': int(
                    ck['best_epoch']
                ),
                'best_validation_duration_norm_mse':
                    float(
                        ck[
                            'best_validation_duration_norm_mse'
                        ]
                    ),
                'checkpoint': str(
                    p['best']
                ),
            })

BEST_CHECKPOINTS = pd.DataFrame(
    best_checkpoint_records
)
BEST_CHECKPOINTS.to_csv(
    TABLE_DIR / 'NB05_best_checkpoint_summary.csv',
    index=False
)

manifest = {
    'notebook':
        '05_unified_nn_core_ablation_study_PAPER_FINAL',
    'role':
        'controlled Strategy-III ablation extension; canonical paper model unchanged',
    'base_dir': str(BASE_DIR),
    'uses_balanced_final': False,
    'dataset_dir': str(DATA_DIR),
    'canonical_training_run':
        str(CANONICAL_TRAIN_RUN),
    'notebook03_run':
        str(NOTEBOOK03_RUN),
    'shared_split_sha256':
        SHARED_SPLIT_SHA256,
    'random_seed':
        RANDOM_SEED,
    'canonical_architecture': {
        'hidden_dims':
            NN_HIDDEN_DIMS,
        'dropout':
            NN_DROPOUT,
        'output':
            'single directed temperature rate',
        'duration_head':
            False,
        'slope_loss':
            False,
    },
    'optimizer': {
        'name':
            'AdamW',
        'learning_rate':
            NN_LR,
        'weight_decay':
            NN_WEIGHT_DECAY,
        'gradient_clip':
            5.0,
        'max_epochs':
            NN_MAX_EPOCHS,
        'validation_every_epochs':
            NN_VALIDATION_EVERY,
        'early_stop_patience_validations':
            NN_EARLY_STOP_PATIENCE_VALIDATIONS,
        'scheduler_patience_validations':
            NN_SCHEDULER_PATIENCE_VALIDATIONS,
        'scheduler_factor':
            NN_SCHEDULER_FACTOR,
        'minimum_lr':
            NN_MIN_LR,
        'minimum_improvement':
            NN_MIN_DELTA,
        'canonical_global_optimizer_steps_per_epoch':
            CANONICAL_GLOBAL_STEPS_PER_EPOCH,
    },
    'canonical_phase_balance': {
        'heating':
            0.5,
        'cooling':
            0.5,
    },
    'inverse_frequency_support_weights':
        False,
    'ablations':
        ABLATION_SPECS,
    'canonical_replay': {
        'strict':
            STRICT_CANONICAL_REPLAY,
        'tolerance_min':
            CANONICAL_REPLAY_TOL_MIN,
        'max_abs_difference_min':
            max_replay_diff,
    },
    'primary_test_support':
        'exact common TEST anchors from frozen Notebook 03',
    'secondary_test_support':
        'full TEST anchors for all NN variants',
    'bootstrap': {
        'unit':
            'physical event',
        'replicates':
            N_BOOTSTRAP,
        'seed':
            BOOTSTRAP_SEED,
        'CI_level':
            CI_LEVEL,
    },
    'best_checkpoints':
        BEST_CHECKPOINTS.to_dict(
            orient='records'
        ),
    'key_outputs': {
        'primary_metrics':
            str(
                TABLE_DIR /
                'PAPER_NB05_ablation_duration_metrics_EXACT_COMMON_SUPPORT.csv'
            ),
        'full_test_metrics':
            str(
                TABLE_DIR /
                'PAPER_NB05_ablation_duration_metrics_FULL_TEST_SUPPORT.csv'
            ),
        'ablation_deltas':
            str(
                TABLE_DIR /
                'PAPER_NB05_ablation_deltas_vs_canonical.csv'
            ),
        'ablation_bootstrap':
            str(
                TABLE_DIR /
                'PAPER_NB05_ablation_paired_event_cluster_bootstrap_CI.csv'
            ),
        'trajectory_metrics':
            str(
                TABLE_DIR /
                'PAPER_NB05_ablation_TEST_trajectory_metrics.csv'
            ),
    },
    'interpretation_guardrail': (
        'Ablation deltas identify sensitivity to the tested controlled '
        'intervention within this PAPER-FINAL implementation. They do not '
        'establish universal causal effects beyond the monitored building.'
    ),
}

with open(
    OUT_ROOT / 'NB05_core_ablation_manifest.json',
    'w',
    encoding='utf-8'
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
        default=str
    )

status = pd.DataFrame([
    {
        'check':
            'Uses PAPER-TEMP rather than PAPER-TEMP-NEW',
        'status':
            'PASS',
    },
    {
        'check':
            'Shared split matches Notebook 01/02/03',
        'status':
            'PASS',
    },
    {
        'check':
            'Canonical full feature order reproduced',
        'status':
            'PASS',
    },
    {
        'check':
            'Canonical preprocessing reproduced',
        'status':
            'PASS',
    },
    {
        'check':
            'Canonical hard Notebook-03 predictions replayed',
        'status':
            'PASS',
    },
    {
        'check':
            'No BALANCED_FINAL support-frequency weights',
        'status':
            'PASS',
    },
    {
        'check':
            'No room identity / no duration head / no slope loss',
        'status':
            'PASS',
    },
    {
        'check':
            'TEST excluded from checkpoint selection',
        'status':
            'PASS',
    },
])

status.to_csv(
    AUDIT_DIR / 'NB05_final_integrity_checks.csv',
    index=False
)

print('\nFINAL NOTEBOOK-05 INTEGRITY STATUS')
display(status)

print('\nBest checkpoints:')
display(BEST_CHECKPOINTS)

print('\nOutput root:', OUT_ROOT)


## Interpretation

A positive paired duration-error difference supports a benefit of the removed component in the tested configuration. An interval containing zero leaves the difference uncertain; it does not establish equivalence. Interpret duration changes alongside the secondary temperature/rate results.

The duration-gradient comparison retains timing-based validation selection. The derivative-history comparison retains current derivative estimates. Separate-mode networks change capacity and validation selection as well as sharing. These qualifications limit the attribution of any observed difference.

The variants are not each repeated across the five training seeds used in Notebook 07. That later experiment assesses the reference model's training stability, rather than the seed stability of every component effect.
